# AI 동적 플래너 — 최종 제출본

자연어로 계획을 말하면 필요한 조건만 질문하고, 고정 일정을 피해 실행 가능한 계획을 생성하며 실패한 작업을 다시 배치하는 LangChain 기반 서비스입니다.

- **대상 사용자:** 혼자 계획을 구체화하거나 실패 후 조정하기 어려운 사람
- **해결하려는 문제:** 사용자가 해야 할 일만 알고 있어도 필요한 조건을 보완해 실행 가능한 일정으로 만들고, 계획이 어긋났을 때 남은 작업을 다시 배치합니다.
- **핵심 흐름:** `입력 → 컨텍스트 구조화 → 조건부 질문 → LLM 순서 판단 → Python 시간 배치·검증 → 출력·재계획`
- **LLM이 필요한 이유:** 일정 표현은 자유롭고 목표·실행 작업·반복 조건·선호 시간의 의미를 구분해야 하므로 모든 문장 패턴을 규칙만으로 처리하기 어렵습니다.
- **Python을 함께 쓰는 이유:** 날짜 계산과 시간 충돌은 언어모델보다 결정적 계산이 정확합니다.


In [1]:
%pip install -q langchain langchain-openai python-dotenv pydantic gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.9/125.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 15.1 MB/s eta 0:00:00


## 1. 데이터 모델과 Structured Output

Pydantic Structured Output으로 LLM 출력을 `PlanningContext`, `PlanningDecision`, `PlanUpdate` 스키마에 맞춥니다. 이 컴포넌트가 없으면 모델이 필드명이나 날짜 형식을 자유롭게 바꿔 후속 일정 계산이 불안정해집니다.


In [2]:
from typing import Literal
from pydantic import BaseModel, Field


class Task(BaseModel):
    id: str = Field(description="작업을 구별하기 위한 고유 ID")
    title: str = Field(description="할 일의 이름")

    estimated_minutes: int | None = Field(
        default=None,
        description="작업 1회당 예상 소요 시간"
    )

    estimate_source: Literal["user", "suggested", "unknown"] = Field(
        default="unknown",
        description="예상 시간이 사용자 입력인지 AI 임시 제안인지 표시"
    )

    deadline: str | None = Field(
        default=None,
        description="YYYY-MM-DD 형식의 마감일"
    )

    available_from: str | None = Field(
        default=None,
        description="이 작업을 배치할 수 있는 최초 날짜. 내부 일정 계산에도 사용"
    )

    priority: int | None = Field(
        default=None,
        description="1이 가장 높은 우선순위"
    )

    splittable: bool | None = Field(
        default=None,
        description="작업을 여러 시간대로 나눌 수 있는지 여부"
    )

    is_recurring: bool = Field(
        default=False,
        description="계획 기간 동안 반복 수행하는 작업인지 여부"
    )

    frequency_per_week: int | None = Field(
        default=None,
        description="반복 작업의 주간 수행 횟수. 일회성 작업이면 null"
    )

    preferred_period: Literal["morning", "afternoon", "evening"] | None = Field(
        default=None,
        description="사용자가 작업을 선호하는 시간대"
    )


class RecurrenceRule(BaseModel):
    frequency: Literal["daily", "weekly", "monthly"] = Field(
        description="반복 단위"
    )
    interval: int = Field(
        default=1,
        ge=1,
        description="몇 단위마다 반복하는지. 매주는 1, 격주는 2"
    )
    weekdays: list[int] = Field(
        default_factory=list,
        description="반복 요일. 월요일=0부터 일요일=6까지"
    )
    day_of_month: int | None = Field(
        default=None,
        ge=1,
        le=31,
        description="매월 반복되는 날짜"
    )
    until: str | None = Field(
        default=None,
        description="반복 종료일. YYYY-MM-DD 형식"
    )
    count: int | None = Field(
        default=None,
        ge=1,
        description="반복 횟수"
    )


class FixedEvent(BaseModel):
    id: str = Field(description="고정 일정을 구별하기 위한 고유 ID")
    title: str
    date: str | None = Field(
        default=None,
        description="단일 일정 날짜 또는 반복 일정의 시작 기준일"
    )
    end_date: str | None = Field(
        default=None,
        description="여러 날 일정의 종료 날짜. 하루 일정이면 null"
    )
    is_all_day: bool = Field(
        default=False,
        description="시간 없이 날짜 범위 전체를 차지하는 일정인지 여부"
    )
    start_time: str | None = None
    end_time: str | None = None
    recurrence: RecurrenceRule | None = Field(
        default=None,
        description="반복하지 않는 일정이면 null"
    )


class AvailableSlot(BaseModel):
    id: str = Field(description="가용 시간대를 구별하기 위한 고유 ID")
    date: str | None = Field(
        default=None,
        description="특정 날짜에만 가능한 경우 YYYY-MM-DD"
    )
    weekdays: list[int] = Field(
        default_factory=list,
        description="반복 가능한 요일. 월요일=0부터 일요일=6까지"
    )
    start_time: str | None = Field(
        default=None,
        description="HH:MM 형식의 시작 시간"
    )
    end_time: str | None = Field(
        default=None,
        description="HH:MM 형식의 종료 시간"
    )
    available_minutes: int | None = Field(
        default=None,
        ge=1,
        description="정확한 시간대 없이 분량만 말한 경우의 가용 시간"
    )


class PlanningContext(BaseModel):
    goal: str
    start_date: str | None = None
    end_date: str | None = None
    tasks: list[Task]
    fixed_events: list[FixedEvent] = Field(default_factory=list)
    available_slots: list[AvailableSlot] = Field(default_factory=list)
    unavailable_weekdays: list[int] = Field(
        default_factory=list,
        description="사용자가 계획할 수 없다고 명시한 반복 요일. 월요일=0"
    )
    planning_strategy: Literal["quick", "balanced", "buffer"] = Field(
        default="quick",
        description="빠른 완료, 균등 분산, 여유일 확보 중 배치 전략"
    )
    missing_information: list[str] = Field(default_factory=list)


class ScheduleItem(BaseModel):
    task_id: str
    title: str
    date: str
    start_time: str
    end_time: str
    minutes: int
    status: Literal["planned", "completed", "missed"] = "planned"


class Plan(BaseModel):
    schedule: list[ScheduleItem]
    warnings: list[str] = Field(default_factory=list)
    unscheduled_tasks: list[Task] = Field(
        default_factory=list,
        description="가용 시간 부족으로 아직 배치하지 못한 작업 인스턴스"
    )
    explanation: str


class PlanningDecision(BaseModel):
    ordered_task_ids: list[str] = Field(
        description="먼저 배치할 작업부터 나열한 작업 ID"
    )
    explanation: str = Field(
        description="이 순서로 계획한 이유를 사용자가 이해하기 쉽게 설명"
    )


class PlanValidation(BaseModel):
    is_valid: bool
    errors: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)


class PlanUpdate(BaseModel):
    completed_task_ids: list[str] = Field(default_factory=list)
    missed_task_ids: list[str] = Field(default_factory=list)
    new_fixed_events: list[FixedEvent] = Field(default_factory=list)
    changed_priorities: dict[str, int] = Field(default_factory=dict)
    additional_notes: str = ""

#추가 질문 생성
class FollowUpQuestion(BaseModel):
    key: str = Field(
        description="질문으로 확인하려는 정보의 식별자"
    )
    question: str = Field(
        description="사용자에게 보여줄 자연스러운 질문"
    )
    example: str | None = Field(
        default=None,
        description="사용자가 참고할 수 있는 짧은 답변 예시"
    )
class FollowUpQuestions(BaseModel):
    questions: list[FollowUpQuestion]


## 2. Python 후처리: 반복 일정, 빈 시간 계산, 검증

LLM은 자연어 의미와 작업 순서를 판단하고, Python은 다음을 결정적으로 처리합니다.

- `다음 주 금요일` 같은 상대 날짜 원문 재검증
- 반복 약속과 반복 작업의 실제 날짜 확장
- 종일·기간 일정이 차지하는 가용 슬롯 제거
- 작업 배치, 시간 겹침, 고정 일정 충돌 검증
- 미배치 작업 별도 보존

이 후처리가 없으면 그럴듯하지만 실제로 겹치는 일정이 출력될 수 있습니다.


## 3. 프롬프트 설계

역할과 목적이 다른 작업을 하나의 거대한 프롬프트로 처리하지 않고 네 단계로 분리합니다.

1. **구조화 프롬프트:** 오늘 날짜, 목표와 행동의 구분, 날짜·시간 형식, 반복 규칙을 제공합니다.
2. **후속 질문 프롬프트:** Python이 계산한 누락 항목만 전달해 이미 말한 내용을 다시 묻지 않게 합니다.
3. **계획 프롬프트:** 구조화된 작업과 제약을 보고 의미적인 선후관계와 이유만 판단합니다.
4. **재계획 프롬프트:** 변경 내용을 구조화하되 실제 시간 재배치는 Python이 담당합니다.

예시와 출력 형식을 프롬프트에 포함해 답변의 일관성을 높였습니다.


In [3]:
"""AI 동적 플래너에서 사용하는 모든 ChatPromptTemplate 정의."""

from langchain_core.prompts import ChatPromptTemplate


extract_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
오늘 날짜는 {current_date}이다.
사용자의 할 일과 계획 조건을 분석한다.

규칙:
- 목표와 실행 작업을 구분한다.
- goal에는 사용자가 달성하고 싶은 모든 목표를 빠짐없이 포함한다.
  여러 목표가 있으면 하나만 선택하지 말고 모두 요약한다.
- '5kg 감량', '취업 준비', '프로젝트 완성'처럼 결과를 나타내는 표현은
  goal에 기록하고 그대로 tasks에 넣지 않는다.
- tasks에는 달력에 배치할 수 있는 구체적인 행동만 기록한다.
  예: 30분 걷기, 식단 기록, 주제 선정, 핵심 기능 구현.
- 작업 제목에는 소요 시간이나 반복 횟수를 넣지 않고 행동 이름만 기록한다.
  예: '40분 걷기'가 아니라 title='걷기', estimated_minutes=40으로 기록한다.
- 사용자가 목표만 말하고 구체적인 행동을 정하지 않았다면 tasks를 비워 두고,
  어떤 활동이나 작업을 할지 missing_information에 추가한다.
- 사용자가 말하지 않은 시간이나 마감일을 임의로 만들지 않는다.
- 각 작업에 고유한 ID를 부여한다.
- 계획 생성에 필요한 정보가 없으면 missing_information에 기록한다.
- 최소한 작업별 예상 시간, 계획 기간, 가용 시간은 확인해야 한다.
- 날짜는 가능하면 YYYY-MM-DD 형식으로 정리한다.
- 이번 주, 다음 주, 수요일 같은 상대 날짜는 오늘 날짜를 기준으로 계산한다.
- '다음 주 금요일'은 이번 주 금요일이 아니라 다음 달력 주의 금요일이다.
- 사용자가 말하지 않은 시간은 임의로 만들지 않는다.
- 사용자가 구체적인 시간을 말하지 않았다면 시간을 추측하지 않는다.
- '오전', '오후', '저녁'만 입력된 경우 start_time과 end_time은 null로 둔다.
- 시간이 필요한 경우 missing_information에 추가한다.
- 고정 일정의 시작 시간이나 종료 시간이 없으면 missing_information에 추가한다.
- '저녁', '오후' 같은 표현을 임의의 구체적인 시간으로 바꾸지 않는다.
- 날짜 범위 전체를 차지하는 출장·여행 같은 일정은 is_all_day=true,
  date=시작일, end_date=종료일로 기록하고 시간은 null로 둔다.
- 시작·종료 시간이 명시된 일정은 is_all_day=false로 기록한다.
- 운동, 공부, 기록처럼 반복 수행하는 작업은 is_recurring=true로 기록한다.
- 주제 선정, UI 구현, 테스트처럼 한 번 완료하는 작업은 is_recurring=false로 기록한다.
- 일회성 작업은 frequency_per_week를 null로 둔다.
- 반복 작업의 수행 빈도를 사용자가 말하지 않았다면 frequency_per_week를 null로 둔다.
- frequency_per_week는 일주일 기준 횟수이다. '매일'은 7, '주 3회'는 3이다.
- 고정 반복 일정과 AI가 배치해야 하는 반복 작업을 구분한다.
- '운동 주 3회', '공부 주 5회'처럼 시간이 정해지지 않은 활동은
  Task.frequency_per_week에 기록하고 FixedEvent로 만들지 않는다.
- 이미 날짜나 시간 규칙이 정해진 약속, 수업, 회의만 FixedEvent로 기록한다.
- 반복 일정을 날짜별 이벤트 여러 개로 생성하지 않고 recurrence에 구조화한다.
- 반복이 없는 단일 일정은 recurrence를 null로 두고 date에 날짜를 기록한다.
- '매일'은 frequency=daily, interval=1이다.
- '3일마다'는 frequency=daily, interval=3이다.
- '매주 수요일'은 frequency=weekly, interval=1, weekdays=[2]이다.
- '격주 화요일'은 frequency=weekly, interval=2, weekdays=[1]이다.
- '평일마다'는 frequency=weekly, interval=1, weekdays=[0,1,2,3,4]이다.
- '매달 15일'은 frequency=monthly, interval=1, day_of_month=15이다.
- 사용자가 반복 종료일이나 횟수를 말하지 않았다면 until과 count는 null로 둔다.
- 반복 일정의 기본 종료 범위는 전체 계획 기간이다.
- '수요일에 약속'처럼 반복 표현이 없으면 가장 가까운 수요일의 단일 일정이다.
- 전체 계획 기간이 이미 있고 사용자가 별도의 작업 기간을 말하지 않았다면,
  개별 작업도 전체 계획 기간을 따른다. 같은 기간을 다시 누락 정보로 요청하지 않는다.
- 건강 목표에 대해 식단, 열량, 감량 속도, 운동 강도를 임의로 처방하지 않는다.
- 가용 시간은 가능하면 평일과 주말 또는 요일별로 구분해 확인한다.
- 가용 시간은 available_slots 목록에 저장한다. 임의의 키를 가진 딕셔너리를 만들지 않는다.
- '평일 저녁 7시부터 9시'는 weekdays=[0,1,2,3,4],
  start_time='19:00', end_time='21:00'인 하나의 AvailableSlot이다.
- '토요일 오후 2시부터 5시'는 weekdays=[5],
  start_time='14:00', end_time='17:00'인 AvailableSlot이다.
- 사용자가 '평일 하루 2시간'처럼 분량만 말하면 available_minutes=120으로 기록하고,
  start_time과 end_time은 임의로 만들지 않는다.
- 가용 시간의 시작 또는 종료 시각이 없으면 missing_information에 추가한다.
- '매주 토요일은 안 돼', '금요일은 불가능'처럼 사용할 수 없는 요일은
  available_slots에 넣지 말고 unavailable_weekdays에 요일 숫자로 기록한다.
- 요일 숫자는 월요일=0, 화요일=1, 수요일=2, 목요일=3,
  금요일=4, 토요일=5, 일요일=6이다.
- 사용자가 작업 예상 시간을 명시하면 estimate_source='user'로 기록한다.
- 사용자가 예상 시간을 말하지 않았다면 estimated_minutes=null,
  estimate_source='unknown'으로 두며 임의로 시간을 만들지 않는다.
- '오전에 공부', '저녁에 운동'처럼 작업별 선호 시간대가 있으면
  preferred_period에 morning, afternoon, evening 중 하나로 기록한다.
"""
    ),
    ("human", "{user_input}"),
])


question_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 사용자가 부담 없이 계획을 세울 수 있도록 돕는 친절한 플래너이다.
계획을 만드는 데 부족한 정보를 자연스러운 대화형 질문으로 확인한다.

질문 작성 규칙:
- question_needs에 전달된 항목만 질문하고 새로운 주제를 임의로 추가하지 않는다.
- QuestionNeed 하나당 질문 하나를 만든다.
- 같은 주제에 포함된 세부 정보는 한 질문으로 묶는다.
- 시작일과 종료일은 계획 기간 질문 하나로 묻는다.
- 평일과 주말 가용 시간은 질문 하나로 묻는다.
- 고정 일정의 시작과 종료 시간은 질문 하나로 묻는다.
- 설문조사나 데이터 입력 양식처럼 딱딱하게 질문하지 않는다.
- '분 단위로 입력하세요'처럼 특정 단위를 강요하지 않는다.
- 시간은 '대략 얼마나 걸릴까요?'처럼 자연스럽게 질문한다.
- 사용자가 정확한 시간을 몰라도 답할 수 있게 한다.
- 각 질문 뒤에 짧은 답변 예시를 제공한다.
- 질문 목록 전체에서 부담을 낮추는 안내는 한 번만 제공하므로,
  각 질문에 '정확하지 않아도 괜찮아요'를 반복하지 않는다.
- 이미 제공된 정보는 다시 질문하지 않는다.
- 작업 예상 소요 시간은 질문하지 않는다.
- 목표가 실행 가능한 작업으로 구체화되지 않았다면 소요 시간보다
  구체적으로 어떤 활동이나 작업을 할지 먼저 질문한다.
- 사용자가 말하지 않은 주간 실행 횟수를 임의로 정하지 않고 질문한다.
- 평일과 주말의 가용 시간대를 한 질문에서 항목별로 묻는다.
- 고정 일정의 시간이 비어 있으면 반드시 질문한다.
- 반복 일정은 개별 날짜를 나열하지 말고 반복 규칙을 자연스럽게 표현해 질문한다.
  예: '매주 수요일 약속은 보통 몇 시부터 몇 시까지인가요?'
- 건강 목표는 사용자가 이미 생각한 활동을 묻는 수준으로 제한한다.
  특정 식단, 섭취 열량, 감량 속도 또는 운동 강도를 추천하거나 처방하지 않는다.
- 한 번에 최대 4개까지만 질문한다.
- 계획 초안 생성에 반드시 필요한 정보만 질문한다.
- 작업 예상 소요 시간은 사용자가 이미 알고 있을 때만 확인하고,
  모른다고 답한 항목을 같은 방식으로 다시 질문하지 않는다.
- 예상 시간을 모르면 이후 단계에서 수정 가능한 임시값을 제안하므로
  예상 시간만을 얻기 위한 추가 질문은 만들지 않는다.
- 현재 질문 라운드가 2차라면 같은 문장을 반복하지 말고,
  사용자가 고르기 쉬운 짧은 예시나 선택지를 제시한다.
- 가장 중요한 질문부터 배치한다.
- 사용자의 답변은 이후 별도 단계에서 분 단위 데이터로 변환되므로,
  질문에서 분 단위 답변을 강요하지 않는다.

좋은 질문 예시:
- 체중 감량을 위해 이번 달에 실천하고 싶은 활동은 무엇인가요?
  예: 주 3회 걷기, 식사 기록처럼 이미 생각해 둔 활동
- AI 미니 프로젝트를 완성하기 위해 어떤 작업이 필요할까요?
  예: 주제 선정, 핵심 기능 개발, UI 구현, 테스트
- 이 계획은 언제부터 언제까지 진행할까요?
  예: 오늘부터 한 달, 10월 1일부터 10월 31일까지
- 평일과 주말에는 각각 언제 계획을 실행할 수 있나요?
  예: 평일 저녁 8시~11시, 토요일 오후 1시~6시
- 수요일 약속은 대략 몇 시부터 몇 시까지인가요?

나쁜 질문 예시:
- 예상 소요 시간을 분 단위로 입력하세요.
- 하루 가용 시간을 분 단위로 알려주세요.
"""
    ),
    (
        "human",
        """
현재까지 파악한 계획 정보:
{context}

반드시 확인해야 하는 질문 항목:
{question_needs}

현재 질문 라운드:
{round_number}

이전에 질문한 항목:
{asked_keys}
"""
    ),
])


PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
당신은 현실적인 일정의 우선순위를 판단하는 AI 플래너이다.
Python이 실제 시간 계산을 담당하므로 시간을 직접 만들지 않는다.
주어진 작업 ID만 사용하고 모든 작업 ID를 정확히 한 번씩 포함한다.
마감일이 빠른 작업, 우선순위 숫자가 작은 작업, 선행 작업을 먼저 둔다.
요일 숫자는 월요일=0, 화요일=1, 수요일=2, 목요일=3, 금요일=4, 토요일=5, 일요일=6이다.
설명에서 고정 일정의 요일이나 시간을 언급할 때는 주어진 정보를 정확히 따른다.
설명에는 사용자의 목표와 제약을 고려한 핵심 이유만 간결하게 쓴다.
설명에서 내부 작업 ID나 회차 번호를 나열하지 않는다.
"""),
    ("human", "목표: {goal}\n계획 기간: {period}\n작업: {tasks}\n고정 일정: {events}"),
])


REPLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
사용자의 계획 변경 내용을 PlanUpdate로 구조화한다.
이전 일정에 있는 정확한 task_id를 사용한다.
'못했다', '미완료', '실패했다'고 말한 작업만 missed_task_ids에 넣는다.
새 약속은 FixedEvent로 만들고 날짜는 YYYY-MM-DD, 시간은 HH:MM으로 기록한다.
운동, 공부, 독서처럼 시간을 옮겨야 하는 활동은 new_fixed_events가 아니다.
날짜와 시작·종료 시간이 모두 명시된 변경 불가능한 약속만 new_fixed_events에 넣는다.
사용자가 말하지 않은 완료 작업이나 새 일정을 만들지 않는다.
상대 날짜는 오늘 날짜를 기준으로 계산한다.
"""),
    ("human", "오늘: {today}\n이전 일정: {schedule}\n변경 내용: {change_text}"),
])


In [4]:
from calendar import monthrange
from datetime import date, datetime, time, timedelta



def _parse_date(value: str) -> date:
    return date.fromisoformat(value)


def _add_months(value: date, months: int) -> date:
    month_index = value.year * 12 + value.month - 1 + months
    year, month_zero_based = divmod(month_index, 12)
    month = month_zero_based + 1
    day = min(value.day, monthrange(year, month)[1])
    return date(year, month, day)


def expand_fixed_event(
    event: FixedEvent,
    plan_start: str,
    plan_end: str,
) -> list[FixedEvent]:
    """고정 일정의 반복 규칙을 계획 기간 안의 실제 날짜 목록으로 확장한다."""
    start = _parse_date(plan_start)
    end = _parse_date(plan_end)

    if end < start:
        raise ValueError("plan_end는 plan_start보다 빠를 수 없습니다.")

    if event.recurrence is None:
        if event.date is None:
            raise ValueError(f"단일 일정 '{event.title}'에 날짜가 없습니다.")
        event_date = _parse_date(event.date)
        return [event] if start <= event_date <= end else []

    rule = event.recurrence
    if rule.until is not None:
        end = min(end, _parse_date(rule.until))

    anchor = max(start, _parse_date(event.date)) if event.date else start
    occurrences: list[date] = []

    def append_if_allowed(candidate: date) -> bool:
        if candidate < start or candidate > end:
            return False
        occurrences.append(candidate)
        return rule.count is not None and len(occurrences) >= rule.count

    if rule.frequency == "daily":
        current = anchor
        while current <= end:
            if append_if_allowed(current):
                break
            current += timedelta(days=rule.interval)

    elif rule.frequency == "weekly":
        if not rule.weekdays:
            raise ValueError(f"주간 반복 일정 '{event.title}'에 요일이 없습니다.")

        current = anchor
        while current <= end:
            days_from_anchor = (current - anchor).days
            week_index = days_from_anchor // 7
            if week_index % rule.interval == 0 and current.weekday() in rule.weekdays:
                if append_if_allowed(current):
                    break
            current += timedelta(days=1)

    elif rule.frequency == "monthly":
        if rule.day_of_month is None:
            raise ValueError(f"월간 반복 일정 '{event.title}'에 날짜가 없습니다.")

        month_cursor = date(anchor.year, anchor.month, 1)
        while month_cursor <= end:
            last_day = monthrange(month_cursor.year, month_cursor.month)[1]
            if rule.day_of_month <= last_day:
                candidate = date(
                    month_cursor.year,
                    month_cursor.month,
                    rule.day_of_month,
                )
                if candidate >= anchor and append_if_allowed(candidate):
                    break
            month_cursor = _add_months(month_cursor, rule.interval)

    return [
        FixedEvent(
            id=f"{event.id}-{occurrence.isoformat()}",
            title=event.title,
            date=occurrence.isoformat(),
            end_date=None,
            is_all_day=event.is_all_day,
            start_time=event.start_time,
            end_time=event.end_time,
            recurrence=None,
        )
        for occurrence in occurrences
    ]


def expand_fixed_events(
    events: list[FixedEvent],
    plan_start: str,
    plan_end: str,
) -> list[FixedEvent]:
    """여러 고정 일정을 날짜순으로 확장한다."""
    expanded = [
        occurrence
        for event in events
        for occurrence in expand_fixed_event(event, plan_start, plan_end)
    ]
    return sorted(expanded, key=lambda item: (item.date or "", item.start_time or ""))


def _to_datetime(day: date, value: str) -> datetime:
    return datetime.combine(day, time.fromisoformat(value))


def build_free_slots(
    available_slots: list[AvailableSlot], fixed_events: list[FixedEvent],
    plan_start: str, plan_end: str,
) -> list[tuple[datetime, datetime]]:
    """가용 시간을 날짜별로 펼치고 고정 일정과 겹치는 부분을 제거한다."""
    start, end = _parse_date(plan_start), _parse_date(plan_end)
    event_ranges = []
    for event in expand_fixed_events(fixed_events, plan_start, plan_end):
        if event.date and event.is_all_day:
            start_day = _parse_date(event.date)
            end_day = _parse_date(event.end_date or event.date)
            event_ranges.append((
                datetime.combine(start_day, time.min),
                datetime.combine(end_day + timedelta(days=1), time.min),
            ))
        elif event.date and event.start_time and event.end_time:
            start_day = _parse_date(event.date)
            end_day = _parse_date(event.end_date or event.date)
            event_ranges.append((
                _to_datetime(start_day, event.start_time),
                _to_datetime(end_day, event.end_time),
            ))

    free: list[tuple[datetime, datetime]] = []
    current = start
    while current <= end:
        for slot in available_slots:
            applies = slot.date == current.isoformat() or (
                slot.date is None and current.weekday() in slot.weekdays
            )
            if not applies or not slot.start_time or not slot.end_time:
                continue
            pieces = [(_to_datetime(current, slot.start_time), _to_datetime(current, slot.end_time))]
            for busy_start, busy_end in event_ranges:
                next_pieces = []
                for piece_start, piece_end in pieces:
                    if busy_end <= piece_start or busy_start >= piece_end:
                        next_pieces.append((piece_start, piece_end))
                    else:
                        if piece_start < busy_start:
                            next_pieces.append((piece_start, busy_start))
                        if busy_end < piece_end:
                            next_pieces.append((busy_end, piece_end))
                pieces = next_pieces
            free.extend(piece for piece in pieces if piece[1] > piece[0])
        current += timedelta(days=1)
    return sorted(free)


def expand_task_instances(
    tasks: list[Task], plan_start: str, plan_end: str,
    available_dates: set[date] | None = None,
) -> list[Task]:
    """반복 작업을 실제 가용 날짜 안에서 주별 실행 단위로 펼친다."""
    start, end = _parse_date(plan_start), _parse_date(plan_end)
    instances: list[Task] = []
    for task in tasks:
        if not task.is_recurring:
            instances.append(task.model_copy(deep=True))
            continue
        week_cursor = start - timedelta(days=start.weekday())
        number = 1
        while week_cursor <= end:
            week_start = max(start, week_cursor)
            week_end = min(week_cursor + timedelta(days=6), end)
            eligible_dates = [
                week_start + timedelta(days=offset)
                for offset in range((week_end - week_start).days + 1)
                if available_dates is None
                or week_start + timedelta(days=offset) in available_dates
            ]
            count = min(task.frequency_per_week or 1, len(eligible_dates))
            if count == 1:
                target_dates = [eligible_dates[0]]
            elif count > 1:
                target_dates = [
                    eligible_dates[round(index * (len(eligible_dates) - 1) / (count - 1))]
                    for index in range(count)
                ]
            else:
                target_dates = []
            for target_date in target_dates:
                instance = task.model_copy(deep=True)
                instance.id = f"{task.id}-{number}"
                instance.title = task.title
                instance.available_from = target_date.isoformat()
                instance.deadline = target_date.isoformat()
                instances.append(instance)
                number += 1
            week_cursor += timedelta(days=7)
    return instances


def allocate_tasks(
    tasks: list[Task],
    free_slots: list[tuple[datetime, datetime]],
    strategy: str = "quick",
):
    """LLM이 정한 순서의 작업을 실제 빈 시간에 배치한다."""
    slots = [[start, end] for start, end in free_slots]
    schedule, warnings, unscheduled = [], [], []
    recurring_days: set[tuple[str, date]] = set()
    plan_days = sorted({slot[0].date() for slot in slots})
    usable_days = plan_days[:-1] if strategy == "buffer" and len(plan_days) > 1 else plan_days
    one_time_total = sum(not task.is_recurring for task in tasks)
    one_time_index = 0
    for task in tasks:
        minutes = task.estimated_minutes or 60
        candidates = slots
        if task.preferred_period:
            period_ranges = {
                "morning": (0, 12),
                "afternoon": (12, 17),
                "evening": (17, 24),
            }
            start_hour, end_hour = period_ranges[task.preferred_period]
            candidates = [
                slot for slot in candidates
                if start_hour <= slot[0].hour < end_hour
            ]
        if (
            strategy in {"balanced", "buffer"}
            and usable_days
            and not task.is_recurring
        ):
            denominator = max(one_time_total - 1, 1)
            target_index = round(one_time_index * (len(usable_days) - 1) / denominator)
            target_day = usable_days[target_index]
            candidates = sorted(
                candidates,
                key=lambda slot: (
                    slot[0].date() < target_day,
                    abs((slot[0].date() - target_day).days),
                    slot[0],
                ),
            )
        for slot in candidates:
            if strategy == "buffer" and usable_days and slot[0].date() not in usable_days:
                continue
            if task.available_from and slot[0].date() < _parse_date(task.available_from):
                continue
            if task.deadline and slot[0].date() > _parse_date(task.deadline):
                continue
            parts = task.id.rsplit("-", 1)
            recurring_group = parts[0] if len(parts) == 2 and parts[1].isdigit() else None
            if recurring_group and (recurring_group, slot[0].date()) in recurring_days:
                continue
            if int((slot[1] - slot[0]).total_seconds() // 60) < minutes:
                continue
            item_start = slot[0]
            item_end = item_start + timedelta(minutes=minutes)
            schedule.append(ScheduleItem(
                task_id=task.id, title=task.title, date=item_start.date().isoformat(),
                start_time=item_start.strftime("%H:%M"), end_time=item_end.strftime("%H:%M"),
                minutes=minutes,
            ))
            slot[0] = item_end
            if recurring_group:
                recurring_days.add((recurring_group, item_start.date()))
            break
        else:
            warnings.append(f"'{task.title}' 작업을 배치할 충분한 연속 시간이 없습니다.")
            unscheduled.append(task.model_copy(deep=True))
        if not task.is_recurring:
            one_time_index += 1
    return schedule, warnings, unscheduled


def validate_plan(plan: Plan, free_slots: list[tuple[datetime, datetime]]) -> PlanValidation:
    """계획이 가용 시간 안에 있고 서로 겹치지 않는지 검사한다."""
    errors, ranges = [], []
    for item in plan.schedule:
        day = _parse_date(item.date)
        start, end = _to_datetime(day, item.start_time), _to_datetime(day, item.end_time)
        if int((end - start).total_seconds() // 60) != item.minutes:
            errors.append(f"{item.title}: 시간 길이와 minutes가 다릅니다.")
        if not any(a <= start and end <= b for a, b in free_slots):
            errors.append(f"{item.title}: 가용 시간 밖이거나 고정 일정과 겹칩니다.")
        ranges.append((start, end, item.title))
    ranges.sort()
    for previous, current in zip(ranges, ranges[1:]):
        if current[0] < previous[1]:
            errors.append(f"{previous[2]}와 {current[2]} 일정이 겹칩니다.")
    return PlanValidation(is_valid=not errors, errors=errors, warnings=list(plan.warnings))


## 4. 조건부 질문, 계획 생성 및 재계획

질문은 필수 정보가 부족한 경우에만 최대 4개 생성합니다. 예상 시간이 없으면 매번 질문하지 않고 AI 임시 추정값임을 표시해 초안을 만들 수 있습니다. 계획이 실패하면 영향받기 전 일정은 유지하고 미완료 작업과 이후 일정만 다시 배치합니다. 기간 연장 시에는 기존 반복 횟수를 늘리지 않고 `unscheduled_tasks`에 저장된 미배치 작업만 연장 구간에 넣습니다.


In [5]:
from dataclasses import asdict, dataclass
from datetime import date, timedelta
import re



WEEKDAY_NAMES = {
    0: "월요일",
    1: "화요일",
    2: "수요일",
    3: "목요일",
    4: "금요일",
    5: "토요일",
    6: "일요일",
}

KOREAN_WEEKDAYS = {name: number for number, name in WEEKDAY_NAMES.items()}

AVAILABILITY_DAY_GROUPS = {
    "평일": [0, 1, 2, 3, 4],
    "주말": [5, 6],
    **{name: [number] for name, number in KOREAN_WEEKDAYS.items()},
}


def _to_24_hour(period: str | None, hour: int, minute: int) -> str | None:
    """한국어 오전·오후 표현을 HH:MM으로 변환한다."""
    if minute > 59 or hour > 23:
        return None
    if period == "오전":
        hour = 0 if hour == 12 else hour
    elif period in {"오후", "저녁", "밤"}:
        if hour < 12:
            hour += 12
    return f"{hour:02d}:{minute:02d}"


def _restore_explicit_availability(
    context: PlanningContext,
    source_text: str,
) -> None:
    """LLM 병합에서 빠진 명시적 평일·주말·요일 시간대를 원문으로 복원한다."""
    time_token = (
        r"(?:(오전|오후|저녁|밤)\s*)?"
        r"(\d{1,2})(?::(\d{2}))?\s*시?"
    )
    for label, weekdays in AVAILABILITY_DAY_GROUPS.items():
        pattern = re.compile(
            rf"{label}[^\n,;]{{0,35}}?{time_token}"
            rf"\s*(?:부터|에서|~|～|-)\s*{time_token}(?:\s*까지)?",
            re.IGNORECASE,
        )
        matches = list(pattern.finditer(source_text))
        if not matches:
            continue
        match = matches[-1]
        start_period, start_hour, start_minute, end_period, end_hour, end_minute = (
            match.groups()
        )
        end_period = end_period or start_period
        start_time = _to_24_hour(
            start_period, int(start_hour), int(start_minute or 0)
        )
        end_time = _to_24_hour(
            end_period, int(end_hour), int(end_minute or 0)
        )
        allowed_days = [
            day for day in weekdays if day not in context.unavailable_weekdays
        ]
        if not start_time or not end_time or not allowed_days:
            continue
        existing = next(
            (
                slot for slot in context.available_slots
                if slot.date is None and sorted(slot.weekdays) == allowed_days
            ),
            None,
        )
        if existing:
            existing.start_time = start_time
            existing.end_time = end_time
            existing.available_minutes = None
        else:
            context.available_slots.append(AvailableSlot(
                id=f"text-availability-{label}",
                weekdays=allowed_days,
                start_time=start_time,
                end_time=end_time,
            ))


def apply_text_constraints(
    context: PlanningContext,
    source_text: str,
    reference_date: date | None = None,
) -> PlanningContext:
    """요일 불가와 이번 주·다음 주 날짜를 원문 기준으로 재검증한다."""
    updated = context.model_copy(deep=True)
    reference = reference_date or date.today()
    unavailable = set(updated.unavailable_weekdays)
    for name, number in KOREAN_WEEKDAYS.items():
        patterns = [
            rf"{name}.{{0,12}}(?:안\s*돼|안\s*됨|불가|불가능|어려워|제외)",
            rf"(?:안\s*돼|불가|불가능).{{0,12}}{name}",
        ]
        if any(re.search(pattern, source_text, re.IGNORECASE) for pattern in patterns):
            unavailable.add(number)
    updated.unavailable_weekdays = sorted(unavailable)
    if unavailable:
        for slot in updated.available_slots:
            slot.weekdays = [day for day in slot.weekdays if day not in unavailable]
        for event in updated.fixed_events:
            if event.recurrence and "불가" in event.title:
                event.recurrence.weekdays = [
                    day for day in event.recurrence.weekdays if day in unavailable
                ]

    _restore_explicit_availability(updated, source_text)

    relative_deadline = re.search(
        r"(이번\s*주|다음\s*주)\s*"
        r"(월요일|화요일|수요일|목요일|금요일|토요일|일요일)\s*까지",
        source_text,
    )
    if relative_deadline:
        week_text, weekday_text = relative_deadline.groups()
        monday = reference - timedelta(days=reference.weekday())
        week_offset = 7 if re.sub(r"\s+", "", week_text) == "다음주" else 0
        target = monday + timedelta(
            days=week_offset + KOREAN_WEEKDAYS[weekday_text]
        )
        old_end_date = updated.end_date
        updated.end_date = target.isoformat()
        if updated.start_date is None:
            updated.start_date = reference.isoformat()
        for task in updated.tasks:
            if task.deadline == old_end_date:
                task.deadline = target.isoformat()
    return updated


@dataclass
class QuestionNeed:
    key: str
    topic: str
    details: list[str]
    priority: int

    def to_dict(self) -> dict:
        return asdict(self)


def _describe_event(event) -> str:
    if event.recurrence is None:
        return f"{event.date or '날짜 미정'} {event.title}"

    rule = event.recurrence
    if rule.frequency == "weekly" and rule.weekdays:
        weekdays = "·".join(WEEKDAY_NAMES[day] for day in rule.weekdays)
        interval_text = "매주" if rule.interval == 1 else f"{rule.interval}주마다"
        return f"{interval_text} {weekdays} {event.title}"
    if rule.frequency == "daily":
        interval_text = "매일" if rule.interval == 1 else f"{rule.interval}일마다"
        return f"{interval_text} {event.title}"
    if rule.frequency == "monthly" and rule.day_of_month is not None:
        interval_text = "매달" if rule.interval == 1 else f"{rule.interval}개월마다"
        return f"{interval_text} {rule.day_of_month}일 {event.title}"
    return f"반복 일정 {event.title}"


def describe_available_slot(slot) -> str:
    """내부 슬롯 ID 대신 사용자에게 보여줄 자연스러운 이름을 만든다."""
    if slot.date:
        return f"{slot.date} 가용 시간"
    if slot.weekdays == [0, 1, 2, 3, 4]:
        return "평일"
    if slot.weekdays == [5, 6]:
        return "주말"
    if slot.weekdays:
        return "·".join(WEEKDAY_NAMES[day] for day in slot.weekdays)
    return "가용 시간"


def is_valid_time(value: str | None) -> bool:
    """HH:MM 형식이며 실제 시각으로 유효한지 확인한다."""
    if value is None or re.fullmatch(r"(?:[01]\d|2[0-3]):[0-5]\d", value) is None:
        return False
    return True


def build_question_needs(context: PlanningContext) -> list[QuestionNeed]:
    """현재 상태에서 꼭 확인해야 하는 내용을 질문 주제 단위로 묶는다."""
    needs: list[QuestionNeed] = []

    if context.start_date is None or context.end_date is None:
        missing_period_parts = []
        if context.start_date is None:
            missing_period_parts.append("시작일")
        if context.end_date is None:
            missing_period_parts.append("종료일")
        needs.append(QuestionNeed(
            key="plan_period",
            topic="전체 계획 기간",
            details=missing_period_parts,
            priority=2,
        ))

    if not context.tasks:
        needs.append(QuestionNeed(
            key="tasks",
            topic="목표별 실행 작업",
            details=["각 목표를 위해 실제로 할 활동이나 작업"],
            priority=1,
        ))

    if not context.available_slots:
        needs.append(QuestionNeed(
            key="availability",
            topic="평일과 주말의 가용 시간",
            details=["평일 시간대", "주말 시간대"],
            priority=1,
        ))
    else:
        incomplete_slots = []
        for slot in context.available_slots:
            slot_label = describe_available_slot(slot)
            missing_parts = []
            if slot.date is None and not slot.weekdays:
                missing_parts.append("날짜 또는 요일")
            if not is_valid_time(slot.start_time):
                missing_parts.append("시작 시간")
            if not is_valid_time(slot.end_time):
                missing_parts.append("종료 시간")
            if missing_parts:
                incomplete_slots.append(
                    f"{slot_label}: {', '.join(missing_parts)}"
                )

        if incomplete_slots:
            needs.append(QuestionNeed(
                key="availability_detail",
                topic="불완전한 가용 시간대",
                details=incomplete_slots,
                priority=1,
            ))

    incomplete_events = []
    for event in context.fixed_events:
        event_label = _describe_event(event)
        missing_parts = []
        if event.recurrence is None and event.date is None:
            missing_parts.append("날짜")
        if event.recurrence is not None:
            rule = event.recurrence
            if rule.frequency == "weekly" and not rule.weekdays:
                missing_parts.append("반복 요일")
            if rule.frequency == "monthly" and rule.day_of_month is None:
                missing_parts.append("매월 반복 날짜")
        if not event.is_all_day:
            if not is_valid_time(event.start_time):
                missing_parts.append("시작 시간")
            if not is_valid_time(event.end_time):
                missing_parts.append("종료 시간")
        if missing_parts:
            incomplete_events.append(
                f"{event_label}: {', '.join(missing_parts)}"
            )

    if incomplete_events:
        needs.append(QuestionNeed(
            key="fixed_event_details",
            topic="고정 일정 정보",
            details=incomplete_events,
            priority=1,
        ))

    tasks_without_frequency = [
        task.title
        for task in context.tasks
        if task.is_recurring and task.frequency_per_week is None
    ]
    if tasks_without_frequency:
        needs.append(QuestionNeed(
            key="task_frequency",
            topic="반복 작업의 주간 횟수",
            details=tasks_without_frequency,
            priority=2,
        ))

    return sorted(needs, key=lambda item: item.priority)


def detect_missing_information(context: PlanningContext) -> list[str]:
    """하위 호환용: 필수 질문 주제를 짧은 목록으로 반환한다."""
    return [need.topic for need in build_question_needs(context)]


DEFAULT_ESTIMATED_MINUTES = 60


def apply_suggested_estimates(context: PlanningContext) -> PlanningContext:
    """시간을 모르는 작업에 수정 가능한 임시 예상 시간을 적용한다."""
    updated = context.model_copy(deep=True)

    for task in updated.tasks:
        if task.estimated_minutes is not None:
            if task.estimate_source == "unknown":
                task.estimate_source = "user"
            continue

        task.estimated_minutes = DEFAULT_ESTIMATED_MINUTES
        task.estimate_source = "suggested"

    return updated


def create_plan(context: PlanningContext, model):
    """LLM의 순서 판단과 Python의 시간 계산을 결합해 계획을 만든다."""
    if not context.start_date or not context.end_date:
        raise ValueError("계획 시작일과 종료일이 필요합니다.")
    prepared = apply_suggested_estimates(context)
    free_slots = build_free_slots(
        prepared.available_slots, prepared.fixed_events,
        prepared.start_date, prepared.end_date,
    )
    instance_slots = free_slots
    if prepared.planning_strategy == "buffer" and free_slots:
        last_available_date = max(slot[0].date() for slot in free_slots)
        instance_slots = [
            slot for slot in free_slots if slot[0].date() != last_available_date
        ]
    instances = expand_task_instances(
        prepared.tasks,
        prepared.start_date,
        prepared.end_date,
        available_dates={slot[0].date() for slot in instance_slots},
    )
    chain = PLAN_PROMPT | model.with_structured_output(PlanningDecision, method="function_calling")
    decision = chain.invoke({
        "goal": prepared.goal,
        "period": f"{prepared.start_date}~{prepared.end_date}",
        "tasks": [task.model_dump() for task in instances],
        "events": [event.model_dump() for event in prepared.fixed_events],
    })
    task_by_id = {task.id: task for task in instances}
    ordered = [task_by_id[task_id] for task_id in decision.ordered_task_ids if task_id in task_by_id]
    ordered_ids = {task.id for task in ordered}
    ordered.extend(task for task in instances if task.id not in ordered_ids)
    recurring_tasks = sorted(
        (task for task in ordered if task.is_recurring),
        key=lambda task: (task.available_from or "", task.title, task.id),
    )
    one_time_tasks = [task for task in ordered if not task.is_recurring]
    # 마감이 있는 핵심 일회성 작업을 먼저 확보하고 반복 루틴은 남는 시간에 배치한다.
    ordered = one_time_tasks + recurring_tasks
    schedule, warnings, unscheduled = allocate_tasks(
        ordered, free_slots, strategy=prepared.planning_strategy
    )
    schedule.sort(key=lambda item: (item.date, item.start_time, item.end_time))
    plan = Plan(
        schedule=schedule,
        warnings=warnings,
        unscheduled_tasks=unscheduled,
        explanation=decision.explanation,
    )
    return plan, validate_plan(plan, free_slots)


def replan(context, previous_plan, missed_task_ids, new_fixed_events, model):
    """기존 완료 일정은 유지하고 실패일 이후의 일정을 다시 배치한다."""
    updated = context.model_copy(deep=True)
    updated.fixed_events.extend(new_fixed_events)
    missed_ids = set(missed_task_ids)
    missed_items = [
        item for item in previous_plan.schedule if item.task_id in missed_ids
    ]
    if not missed_items:
        raise ValueError("기존 계획에서 미완료 작업을 찾지 못했습니다.")

    failed_date = min(item.date for item in missed_items)
    failed_retry_date = date.fromisoformat(failed_date) + timedelta(days=1)
    new_event_start_dates = [
        date.fromisoformat(event.date)
        for event in new_fixed_events
        if event.date is not None
    ]
    replan_start = min(
        [failed_retry_date, *new_event_start_dates]
    )
    preserved = [
        item.model_copy(deep=True)
        for item in previous_plan.schedule
        if date.fromisoformat(item.date) < replan_start
        and item.task_id not in missed_ids
    ]
    affected = [
        item for item in previous_plan.schedule
        if item.task_id in missed_ids
        or date.fromisoformat(item.date) >= replan_start
    ]

    base_tasks = {task.id: task for task in updated.tasks}
    remaining_tasks = []
    for item in affected:
        parts = item.task_id.rsplit("-", 1)
        base_id = parts[0] if len(parts) == 2 and parts[1].isdigit() else item.task_id
        base = base_tasks.get(base_id)
        if base is None:
            continue
        task = base.model_copy(deep=True)
        task.id = item.task_id
        task.title = item.title
        task.estimated_minutes = item.minutes
        task.is_recurring = False
        task.frequency_per_week = None
        task.available_from = None
        remaining_tasks.append(task)

    updated.tasks = remaining_tasks
    updated.start_date = replan_start.isoformat()
    replanned, validation = create_plan(updated, model)
    replanned.schedule = sorted(
        preserved + replanned.schedule,
        key=lambda item: (item.date, item.start_time, item.end_time),
    )
    replanned.explanation = (
        f"{len(preserved)}개의 기존 일정은 유지하고, "
        f"미완료 작업과 이후 일정을 {updated.start_date}부터 다시 배치했습니다. "
        f"{replanned.explanation}"
    )
    return replanned, validation


def extend_unplaced_tasks(
    context: PlanningContext,
    previous_plan: Plan,
    new_end_date: str,
    model,
):
    """기존 일정과 반복 횟수는 유지하고 미배치 작업만 연장 구간에 배치한다."""
    if not context.end_date:
        raise ValueError("기존 계획 종료일이 필요합니다.")
    old_end = date.fromisoformat(context.end_date)
    new_end = date.fromisoformat(new_end_date)
    if new_end <= old_end:
        raise ValueError("새 종료일은 기존 계획 종료일보다 늦어야 합니다.")
    if not previous_plan.unscheduled_tasks:
        raise ValueError("연장 구간에 배치할 미배치 작업이 없습니다.")

    extension_context = context.model_copy(deep=True)
    extension_context.start_date = (old_end + timedelta(days=1)).isoformat()
    extension_context.end_date = new_end.isoformat()
    for event in extension_context.fixed_events:
        if (
            event.recurrence
            and event.recurrence.until
            and date.fromisoformat(event.recurrence.until) <= old_end
        ):
            event.recurrence.until = new_end.isoformat()
    extension_context.planning_strategy = "quick"
    extension_context.tasks = []
    for original in previous_plan.unscheduled_tasks:
        task = original.model_copy(deep=True)
        task.is_recurring = False
        task.frequency_per_week = None
        task.available_from = None
        task.deadline = new_end.isoformat()
        extension_context.tasks.append(task)

    extension_plan, _ = create_plan(extension_context, model)
    combined_context = context.model_copy(deep=True)
    combined_context.end_date = new_end.isoformat()
    for event in combined_context.fixed_events:
        if (
            event.recurrence
            and event.recurrence.until
            and date.fromisoformat(event.recurrence.until) <= old_end
        ):
            event.recurrence.until = new_end.isoformat()
    combined_plan = Plan(
        schedule=sorted(
            previous_plan.schedule + extension_plan.schedule,
            key=lambda item: (item.date, item.start_time, item.end_time),
        ),
        warnings=extension_plan.warnings,
        unscheduled_tasks=extension_plan.unscheduled_tasks,
        explanation=(
            f"기존 일정과 반복 작업 횟수는 유지하고, 미배치 작업만 "
            f"{extension_context.start_date}~{new_end_date} 구간에 추가 배치했습니다."
        ),
    )
    free_slots = build_free_slots(
        combined_context.available_slots,
        combined_context.fixed_events,
        combined_context.start_date,
        combined_context.end_date,
    )
    return combined_context, combined_plan, validate_plan(combined_plan, free_slots)


## 5. LangChain 체인 구성

핵심 연결은 `ChatPromptTemplate | model.with_structured_output(...)` 형태의 LCEL 체인입니다.

| 단계 | 컴포넌트 | 없으면 생기는 문제 |
| --- | --- | --- |
| 자연어 구조화 | Prompt + LCEL + Pydantic Structured Output | Python이 모델 출력을 안정적으로 읽지 못함 |
| 조건부 질문 | 별도 질문 Chain | 모든 사용자에게 동일하고 불필요한 질문을 반복함 |
| 작업 순서 | 계획 Chain | 프로젝트의 의미적 선후관계를 일반화하기 어려움 |
| 캘린더 확인 | LangChain `@tool` 2개 | 일정 조회와 충돌 검증을 재사용 가능한 인터페이스로 호출하기 어려움 |

따라서 필수인 프롬프트와 체인 외에 **Structured Output**과 **Tool** 두 종류를 추가로 사용합니다.


In [6]:
from dotenv import load_dotenv
from datetime import date, timedelta
import re
from langchain_openai import ChatOpenAI




load_dotenv(dotenv_path=".env")

model = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

extract_model = model.with_structured_output(PlanningContext, method="function_calling",)

extract_chain = extract_prompt | extract_model

question_model = model.with_structured_output(
    FollowUpQuestions,
    method="function_calling",
)

question_chain = question_prompt | question_model

MAX_QUESTION_ROUNDS = 2
MAX_QUESTIONS_PER_ROUND = 4


def fallback_question(need):
    """질문 생성 LLM이 필수 항목을 빠뜨렸을 때 사용할 안전한 질문."""
    templates = {
        "plan_period": (
            "이 계획은 언제부터 언제까지 진행할까요?",
            "오늘부터 한 달간",
        ),
        "tasks": (
            "각 목표를 위해 실제로 어떤 활동이나 작업을 할 예정인가요?",
            "주 3회 운동, 주제 선정과 핵심 기능 개발",
        ),
        "availability": (
            "평일과 주말에는 각각 언제 계획을 실행할 수 있나요?",
            "평일 저녁 9시~11시, 토요일 낮 12시~6시",
        ),
        "availability_detail": (
            "말씀해 주신 가용 시간의 시작과 종료 시각을 알려주시겠어요?",
            "평일은 밤 9시부터 11시까지",
        ),
        "fixed_event_details": (
            "고정 일정은 언제, 몇 시부터 몇 시까지인가요?",
            "매주 수요일 오후 7시부터 9시까지",
        ),
        "task_frequency": (
            "반복 작업은 일주일에 몇 번 하고 싶으신가요?",
            "운동은 주 3회",
        ),
    }
    question, example = templates.get(
        need.key,
        (f"{need.topic}에 대해 조금 더 알려주시겠어요?", None),
    )
    return FollowUpQuestion(key=need.key, question=question, example=example)

def read_multiline_answer() -> str:
    print("\n질문 번호에 맞춰 답해주세요.")
    print("입력이 끝나면 빈 줄에서 Enter를 누르세요.\n")

    lines = []

    while True:
        line = input()

        if not line.strip():
            break

        lines.append(line)

    return "\n".join(lines)


def pair_questions_and_answers(
    question_texts: list[str],
    answers: str,
) -> str:
    """입력 순서대로 각 질문과 답변을 명시적으로 연결한다."""
    answer_lines = [
        line.strip()
        for line in answers.splitlines()
        if line.strip()
    ]
    numbered_answers: dict[int, list[str]] = {}
    current_number = None
    for line in answer_lines:
        numbered = re.match(r"^(\d+)\s*[.)]\s*(.*)$", line)
        if numbered:
            current_number = int(numbered.group(1))
            numbered_answers.setdefault(current_number, []).append(numbered.group(2))
        elif current_number is not None:
            numbered_answers[current_number].append(line)

    pairs = []
    for index, question in enumerate(question_texts):
        if numbered_answers:
            answer = "\n".join(numbered_answers.get(index + 1, [])) or "답변 없음"
        else:
            answer = answer_lines[index] if index < len(answer_lines) else "답변 없음"
        pairs.append(f"질문: {question}\n답변: {answer}")
    if not numbered_answers and len(answer_lines) > len(question_texts):
        pairs.append(
            "추가 답변(누락하지 말고 관련 질문에 함께 반영):\n"
            + "\n".join(answer_lines[len(question_texts):])
        )
    return "\n\n".join(pairs)

def complete_planning_context(original_input: str) -> PlanningContext:
    """최초 요청을 분석하고, 필요한 경우 답변을 받아 계획 조건을 보완한다."""
    current_date = date.today().isoformat()
    context = extract_chain.invoke({
        "current_date": current_date,
        "user_input": original_input,
    })
    context = apply_text_constraints(
        context, original_input, date.fromisoformat(current_date)
    )
    context.missing_information = detect_missing_information(context)

    print("구조화 결과")
    print(context.model_dump())

    conversation_parts = [f"[최초 요청]\n{original_input}"]
    asked_keys: set[str] = set()

    for round_number in range(1, MAX_QUESTION_ROUNDS + 1):
        question_needs = build_question_needs(context)
        if not question_needs:
            completed = apply_suggested_estimates(context)
            completed.missing_information = []
            print("\n계획 초안 생성에 필요한 정보가 준비됐습니다.")
            print_suggested_estimates(completed)
            return completed

        selected_needs = question_needs[:MAX_QUESTIONS_PER_ROUND]
        questions = question_chain.invoke({
            "context": context.model_dump_json(indent=2),
            "question_needs": [need.to_dict() for need in selected_needs],
            "round_number": round_number,
            "asked_keys": sorted(asked_keys),
        })

        selected_keys = {need.key for need in selected_needs}
        questions.questions = [
            item for item in questions.questions
            if item.key in selected_keys
        ]
        returned_keys = {item.key for item in questions.questions}
        questions.questions.extend(
            fallback_question(need)
            for need in selected_needs
            if need.key not in returned_keys
        )

        print(f"\n추가 질문 {round_number}차")
        print("정확하지 않아도 괜찮으니 현재 생각한 범위에서 답해주세요.\n")

        question_texts = []
        for index, item in enumerate(questions.questions, start=1):
            question_text = f"{index}. {item.question}"
            question_texts.append(question_text)
            print(question_text)

            if item.example:
                clean_example = re.sub(r"^예\s*:\s*", "", item.example.strip())
                print(f"   예: {clean_example}")

        answers = read_multiline_answer()
        paired_answers = pair_questions_and_answers(question_texts, answers)
        conversation_parts.append(
            f"[질문과 사용자 답변 - {round_number}차]\n{paired_answers}"
        )
        asked_keys.update(need.key for need in selected_needs)

        combined_input = f"""
{chr(10).join(conversation_parts)}

[현재까지 구조화된 정보]
{context.model_dump_json(indent=2)}

모든 정보를 종합해 PlanningContext를 다시 작성한다.

병합 규칙:
- 각 질문과 바로 아래 답변을 하나의 쌍으로 해석한다.
- 답변 순서를 다른 질문에 연결하지 않는다.
- 사용자의 가장 최근 답변을 우선한다.
- 최근 답변과 충돌하지 않는 기존 정보는 유지한다.
- 사용자가 답하지 않은 정보는 임의로 만들지 않는다.
- 자연어 시간은 분 단위 정수로 변환한다.
- 목표와 실행 가능한 작업을 구분한다.
- 반복 작업과 고정 반복 일정을 구분한다.
- 기존 고정 일정의 반복 규칙을 유지하고 새 답변의 시간을 반영한다.
- 1시간 반은 90분, 2시간 반은 150분으로 변환한다.
- '하루', '반나절'처럼 실제 작업시간이 불분명한 표현은 분으로 확정하지 않는다.
- 최신 답변에 완전한 시작·종료 시간이 있으면 기존 시간 범위를 모두 교체한다.
"""

        context = extract_chain.invoke({
            "current_date": current_date,
            "user_input": combined_input,
        })
        context = apply_text_constraints(
            context, combined_input, date.fromisoformat(current_date)
        )

        print("\n보완된 구조화 결과")
        context.missing_information = detect_missing_information(context)
        print(context.model_dump())

    remaining_needs = build_question_needs(context)
    if remaining_needs:
        print("\n계획 초안을 만들기 위해 다음 필수 정보가 필요합니다.")
        for need in remaining_needs:
            print(f"- {need.topic}")
        return context

    completed = apply_suggested_estimates(context)
    completed.missing_information = []
    print("\n계획 초안 생성에 필요한 정보가 준비됐습니다.")
    print_suggested_estimates(completed)
    return completed


def print_suggested_estimates(context: PlanningContext) -> None:
    suggested_tasks = [
        task for task in context.tasks
        if task.estimate_source == "suggested"
    ]
    if not suggested_tasks:
        return

    print("\n다음 작업 시간은 초안 생성을 위한 임시 제안입니다.")
    print("계획을 확인한 뒤 언제든 수정할 수 있습니다.")
    for task in suggested_tasks:
        print(f"- {task.title}: {task.estimated_minutes}분 (AI 임시 제안)")


def print_expanded_fixed_events(context: PlanningContext) -> None:
    """계획 기간과 고정 일정 정보가 준비된 경우 실제 날짜를 출력한다."""
    remaining_missing = detect_missing_information(context)
    if remaining_missing or not context.start_date or not context.end_date:
        return

    events = expand_fixed_events(
        context.fixed_events,
        context.start_date,
        context.end_date,
    )

    if not events:
        return

    print("\n계획 기간의 고정 일정")
    for event in events:
        print(
            f"- {event.date} "
            f"{event.start_time}~{event.end_time} "
            f"{event.title}"
        )


def print_plan(title, plan, validation) -> None:
    print(f"\n{title}")
    for item in plan.schedule:
        print(f"- {item.date} {item.start_time}~{item.end_time} {item.title}")
    for warning in plan.warnings:
        print(f"경고: {warning}")
    print(f"설명: {plan.explanation}")
    print(f"검증 결과: {'통과' if validation.is_valid else '실패'}")
    for error in validation.errors:
        print(f"- {error}")


def main() -> None:
    original_input = """
한 달 안에 5kg을 감량하고 싶고
AI 개인 미니 프로젝트도 진행하고 싶어.
매주 수요일 저녁에는 약속이 있어.
"""

    completed_context = complete_planning_context(original_input)
    print_expanded_fixed_events(completed_context)
    if build_question_needs(completed_context):
        return

    plan, validation = create_plan(completed_context, model)
    print_plan("최초 계획", plan, validation)

    # 발표 시연용: 첫 일정 실패와 같은 날의 새 약속을 반영한다.
    if plan.schedule:
        missed_id = plan.schedule[0].task_id
        new_event_date = (
            date.fromisoformat(plan.schedule[0].date) + timedelta(days=1)
        ).isoformat()
        new_event = FixedEvent(
            id="new-event", title="새 약속", date=new_event_date,
            start_time="20:00", end_time="21:00",
        )
        updated_plan, updated_validation = replan(
            completed_context, plan, [missed_id], [new_event], model
        )
        print_plan("재계획", updated_plan, updated_validation)


## 6. 테스트 시나리오 1 — 정보가 부족한 복합 목표

아래 입력은 목표와 반복 약속만 있고 실행 작업, 가용 시간, 정확한 약속 시간이 없습니다. 필요한 질문만 생성한 뒤 답변을 반영해 계획을 만드는지 확인합니다.

권장 답변 예시:

1. `주 3회 운동하고 이번 주 내로 주제 선정을 할 거야`
2. `평일 저녁 9시부터 11시, 토요일 낮 12시부터 6시까지`
3. `오후 8시부터 10시`
4. 계획 기간 질문이 나오면 `오늘부터 한 달간`


In [7]:
test_input_1 = """
한 달 안에 5kg을 감량하고 싶고
AI 개인 미니 프로젝트도 진행하고 싶어.
매주 수요일 저녁에는 약속이 있어.
"""
context_1 = complete_planning_context(test_input_1)

if context_1.available_slots:
    for slot in context_1.available_slots:
        if slot.end_time == '24:00':
            slot.end_time = '23:59'

print_expanded_fixed_events(context_1)
if not build_question_needs(context_1):
    plan_1, validation_1 = create_plan(context_1, model)
    print_plan("테스트 1 계획", plan_1, validation_1)


구조화 결과
{'goal': '5kg 감량, AI 개인 미니 프로젝트 완성', 'start_date': None, 'end_date': None, 'tasks': [], 'fixed_events': [{'id': 'e1', 'title': '수요일 저녁 약속', 'date': '2026-09-23', 'end_date': None, 'is_all_day': False, 'start_time': None, 'end_time': None, 'recurrence': {'frequency': 'weekly', 'interval': 1, 'weekdays': [2], 'day_of_month': None, 'until': '2026-10-22', 'count': None}}], 'available_slots': [], 'unavailable_weekdays': [], 'planning_strategy': 'quick', 'missing_information': ['목표별 실행 작업', '평일과 주말의 가용 시간', '고정 일정 정보', '전체 계획 기간']}

추가 질문 1차
정확하지 않아도 괜찮으니 현재 생각한 범위에서 답해주세요.

1. 5kg 감량과 AI 개인 미니 프로젝트 완성을 위해 구체적으로 어떤 활동이나 작업을 하실 계획인가요?
   예: 주 3회 걷기, 식사 기록, 주제 선정, 핵심 기능 개발
2. 평일과 주말에는 각각 언제쯤 계획을 실행할 수 있나요? 예를 들어 평일 저녁이나 주말 오후 같은 시간대가 있을까요?
   예: 평일 저녁 8시~11시, 주말 오후 1시~6시
3. 매주 수요일 저녁 약속은 보통 몇 시부터 몇 시까지인가요?
   예: 7시부터 9시까지
4. 이 계획은 언제부터 언제까지 진행할까요?
   예: 오늘부터 한 달간

질문 번호에 맞춰 답해주세요.
입력이 끝나면 빈 줄에서 Enter를 누르세요.

주 3회 운동, 식단 조절, 주제 선정
평일 저녁 8시~11시, 주말 오후 1시~6시
오후 7시부터 9시까지
오늘부터 한 달간


보완된 구조

## 7. 테스트 시나리오 2 — 정보가 충분한 프로젝트 입력

작업 시간, 계획 기간, 가용 시간과 약속을 한 번에 제공합니다. 테스트 1과 달리 추가 질문 없이 계획이 생성되는지 확인합니다.


In [8]:
test_input_2 = """
2026년 9월 21일부터 9월 27일까지 LangChain 프로젝트를 준비할 거야.
주제 선정은 60분, 핵심 기능 개발은 120분, 테스트는 60분 걸려.
평일에는 저녁 7시부터 10시까지 가능해.
9월 23일 오후 8시부터 9시까지 약속이 있어.
"""
context_2 = complete_planning_context(test_input_2)
print_expanded_fixed_events(context_2)
if not build_question_needs(context_2):
    plan_2, validation_2 = create_plan(context_2, model)
    print_plan("테스트 2 계획", plan_2, validation_2)


구조화 결과
{'goal': 'LangChain 프로젝트 준비', 'start_date': '2026-09-21', 'end_date': '2026-09-27', 'tasks': [{'id': 'task1', 'title': '주제 선정', 'estimated_minutes': 60, 'estimate_source': 'user', 'deadline': None, 'available_from': None, 'priority': None, 'splittable': None, 'is_recurring': False, 'frequency_per_week': None, 'preferred_period': None}, {'id': 'task2', 'title': '핵심 기능 개발', 'estimated_minutes': 120, 'estimate_source': 'user', 'deadline': None, 'available_from': None, 'priority': None, 'splittable': None, 'is_recurring': False, 'frequency_per_week': None, 'preferred_period': None}, {'id': 'task3', 'title': '테스트', 'estimated_minutes': 60, 'estimate_source': 'user', 'deadline': None, 'available_from': None, 'priority': None, 'splittable': None, 'is_recurring': False, 'frequency_per_week': None, 'preferred_period': None}], 'fixed_events': [{'id': 'event1', 'title': '약속', 'date': '2026-09-23', 'end_date': None, 'is_all_day': False, 'start_time': '20:00', 'end_time': '21:00', 'recurrenc

## 8. 재계획 시연

테스트 1의 첫 작업을 완료하지 못했다고 가정하고 다음 날 새 약속을 추가합니다. 기존 계획 중 영향받지 않는 일정은 유지되고, 실패 작업과 이후 일정만 다시 배치되는지 확인합니다.


In [9]:
if "plan_1" in globals() and plan_1.schedule:
    missed = plan_1.schedule[0]
    new_date = (date.fromisoformat(missed.date) + timedelta(days=1)).isoformat()
    new_event = FixedEvent(
        id="demo-new-event",
        title="새 약속",
        date=new_date,
        start_time="20:00",
        end_time="21:00",
    )
    replanned, revalidation = replan(
        context_1, plan_1, [missed.task_id], [new_event], model
    )
    print("실패한 작업:", missed.title, missed.date)
    print("추가된 약속:", new_date, "20:00~21:00")
    print_plan("재계획 결과", replanned, revalidation)


실패한 작업: 주제 선정 2026-09-22
추가된 약속: 2026-09-23 20:00~21:00

재계획 결과
- 2026-09-22 21:00~22:00 식단 조절
- 2026-09-22 22:00~23:00 운동
- 2026-09-24 20:00~21:00 주제 선정
- 2026-09-24 21:00~22:00 식단 조절
- 2026-09-24 22:00~23:00 운동
- 2026-09-25 20:00~21:00 식단 조절
- 2026-09-25 21:00~22:00 운동
- 2026-09-26 13:00~14:00 식단 조절
- 2026-09-27 13:00~14:00 식단 조절
- 2026-09-28 20:00~21:00 식단 조절
- 2026-09-28 21:00~22:00 운동
- 2026-09-29 20:00~21:00 식단 조절
- 2026-09-29 21:00~22:00 운동
- 2026-10-01 20:00~21:00 식단 조절
- 2026-10-01 21:00~22:00 운동
- 2026-10-02 20:00~21:00 식단 조절
- 2026-10-02 21:00~22:00 운동
- 2026-10-03 13:00~14:00 식단 조절
- 2026-10-04 13:00~14:00 식단 조절
- 2026-10-05 20:00~21:00 식단 조절
- 2026-10-05 21:00~22:00 운동
- 2026-10-06 20:00~21:00 식단 조절
- 2026-10-06 21:00~22:00 운동
- 2026-10-08 20:00~21:00 식단 조절
- 2026-10-08 21:00~22:00 운동
- 2026-10-09 20:00~21:00 식단 조절
- 2026-10-10 13:00~14:00 식단 조절
- 2026-10-11 13:00~14:00 식단 조절
- 2026-10-12 20:00~21:00 식단 조절
- 2026-10-13 20:00~21:00 식단 조절
- 2026-10-15 20:00~21:00 식단 조절
- 202

## 9. 테스트 결과 비교

| 비교 항목 | 테스트 1: 정보 부족 | 테스트 2: 정보 충분 |
| --- | --- | --- |
| 입력 특징 | 복합 목표, 반복 약속만 제시 | 작업·소요 시간·가용 시간·약속을 모두 제시 |
| 추가 질문 | 실행 작업, 가용 시간, 약속 시간, 기간을 필요한 경우에만 질문 | 추가 질문 없이 진행 |
| 구조화 결과 | 운동을 주 3회 반복 작업으로, 프로젝트 작업을 일회성 작업으로 구분 | 주제 선정·핵심 기능 개발·테스트를 개별 작업으로 구분 |
| 일정 생성 | 여러 주의 실제 가용 날짜에 반복 작업 분산 | 선후관계에 따라 같은 주의 빈 시간에 배치 |
| 검증 | 반복 약속과 충돌하지 않는지 검사 | 9월 23일 약속을 피하고 충돌 검사 |

이 비교로 LLM이 입력 내용에 따라 컨텍스트와 질문을 달리 만들고, Python 후처리가 두 경우 모두 같은 기준으로 시간을 검증한다는 점을 확인합니다.

### 재계획에서 확인할 점

- 완료했거나 영향받기 전 작업은 유지됩니다.
- 미완료 작업과 그 이후 작업만 새 약속을 피해서 이동합니다.
- 종일·기간 일정은 해당 날짜의 기존 실행 일정을 제거하고 다시 배치합니다.
- 가용 시간이 부족하면 억지로 넣지 않고 미배치 작업과 해결 UI를 표시합니다.

> 실제 제출본에서는 위 코드 셀을 실행한 출력이 보이는 상태로 저장해야 합니다. API 지연에 대비해 출력 셀을 지우지 않습니다.

## 10. 트러블슈팅

1. **`20:00`을 정수 필드에 넣어 Pydantic 오류 발생**  
   가용 분량과 시간 범위를 하나의 모델로 표현한 것이 원인이었습니다. `AvailableSlot.start_time`, `end_time`, `available_minutes`로 분리했습니다.

2. **“매주 수요일”이 한 날짜로만 저장됨**  
   단일 일정만 표현하던 모델에 `RecurrenceRule`을 추가하고 계획 기간의 실제 날짜로 확장했습니다.

3. **질문이 많고 분 단위 답변을 강요함**  
   누락 필드를 그대로 묻지 않고 관련 항목을 묶어 최대 4개만 질문합니다. 모르는 소요 시간은 AI 임시 추정값으로 초안을 만듭니다.

4. **“다음 주 금요일”, “토요일 불가”를 잘못 해석함**  
   날짜·요일 계산을 LLM에만 맡기지 않고 사용자 원문을 Python 정규식과 날짜 계산으로 다시 검증합니다.

5. **운동 회차가 뒤섞이고 불가능한 날에도 생성됨**  
   전체 반복 횟수를 먼저 만들던 방식을 바꿔 실제 가용 날짜를 기준으로 주별 인스턴스를 생성하고 화면에서는 회차 번호를 숨겼습니다.

6. **재계획에서 기존 작업이 사라지거나 종일 일정과 겹침**  
   영향받기 전 일정을 보존하고 실패 작업 이후만 재배치합니다. 종일 일정은 날짜 범위 전체의 슬롯을 차단합니다.

7. **기간 연장 시 운동·공부 횟수가 추가됨**  
   전체 계획을 다시 생성하지 않고 `Plan.unscheduled_tasks`의 미배치 작업만 사용자가 선택한 연장 구간에 배치합니다.

8. **추가 질문에 적은 주말 시간이 사라짐**  
   여러 줄 답변 중 질문 수를 넘는 줄이 버려지고, LLM이 가용 시간 목록을 다시 쓰면서 주말 슬롯을 누락한 것이 원인이었습니다. 번호가 있는 답변은 다음 번호 전까지 한 답변으로 묶고, 번호가 없는 추가 줄도 보존합니다. 명시적인 평일·주말·요일 시간 범위는 Python이 다시 확인해 누락 슬롯을 복원하며, 결과 화면에 실제 인식한 가용 시간을 표시합니다.

## 11. 한계와 개선 방향

### 현재 한계

- LLM 구조화와 순서 판단은 확률적이어서 표현에 따라 세부 결과가 달라질 수 있습니다.
- 추상적인 목표를 달성하기에 작업 목록이 충분한지까지 보장하지는 않습니다.
- 미입력 작업에는 예시 편향을 피하기 위해 범용 임시값 60분을 적용하므로, 개인의 실제 속도와 작업 난이도 차이를 반영하지 못합니다.
- 긴 작업을 여러 슬롯으로 자동 분할하는 기능은 아직 완성되지 않았습니다.
- 복잡한 작업 의존관계를 별도 그래프로 관리하지 않고 프롬프트 판단에 의존합니다.
- 계획 이력, 완료율, 사용자별 데이터가 영구 저장되지 않습니다.
- 캘린더 Tool은 가상 조회·검증용이며 실제 Google Calendar 쓰기는 제외했습니다.

### 개선 방향

1. 실제 소요 시간과 작업 유형을 학습해 범용 60분 대신 사용자별·작업별 예상 시간을 보정합니다.
2. 작업 의존관계를 DAG로 관리하고 선행 작업 완료 후 후속 작업을 배치합니다.
3. 긴 작업을 최소 블록 단위로 자동 분할합니다.
4. 미배치 경고에서 시간 추가·작업 축소·기간 연장을 바로 선택하게 합니다.
5. 사용자 승인 후 Google Calendar API에 최종 일정을 저장합니다.
6. 계획 버전과 완료율을 데이터베이스에 저장해 재계획 품질을 평가합니다.


## 12. Gradio 서비스 화면

주간 캘린더, 날짜별 상세 계획, 직접 수정, 미완료 작업 재계획, 종일·기간 일정, 미배치 해결 기능을 제공합니다. 노트북 환경에서는 아래 셀을 실행해 웹 화면을 열 수 있습니다.


## 13. 가상 캘린더 Tool

`get_virtual_calendar_events`는 반복·고정 일정을 실제 날짜 목록으로 조회하고, `check_virtual_calendar_conflicts`는 생성된 계획이 가용 시간과 고정 일정에 맞는지 검증합니다. 실제 외부 캘린더를 변경하지 않아 수업 시연에서 안전하게 Tool 사용을 보여줄 수 있습니다.


In [10]:
"""LangChain Tool로 감싼 가상 캘린더 조회·검증 기능."""

import json

from langchain_core.tools import tool



@tool
def get_virtual_calendar_events(context_json: str) -> list[dict]:
    """계획 기간의 고정 일정을 날짜별 가상 캘린더 이벤트로 조회한다."""
    context = PlanningContext.model_validate_json(context_json)
    if not context.start_date or not context.end_date:
        return []
    return [
        event.model_dump()
        for event in expand_fixed_events(
            context.fixed_events, context.start_date, context.end_date
        )
    ]


@tool
def check_virtual_calendar_conflicts(
    context_json: str,
    plan_json: str,
) -> dict:
    """가상 캘린더의 가용 시간·고정 일정과 계획의 충돌을 검사한다."""
    context = PlanningContext.model_validate_json(context_json)
    plan = Plan.model_validate_json(plan_json)
    free_slots = build_free_slots(
        context.available_slots,
        context.fixed_events,
        context.start_date,
        context.end_date,
    )
    return validate_plan(plan, free_slots).model_dump()


In [11]:
"""AI 동적 플래너 Gradio 서비스 화면."""

from datetime import date, datetime, timedelta
from html import escape
from collections import Counter

import gradio as gr
from langchain_core.prompts import ChatPromptTemplate



replan_chain = REPLAN_PROMPT | model.with_structured_output(
    PlanUpdate, method="function_calling"
)


def empty_plan_controls():
    return gr.update(choices=[], value=[]), []


def plan_choices(plan: Plan):
    return [
        f"{item.date} {item.start_time}~{item.end_time} · {item.title} [{item.task_id}]"
        for item in plan.schedule
    ]


def plan_rows(plan: Plan):
    return [
        [item.task_id, item.title, item.date, item.start_time, item.end_time]
        for item in plan.schedule
    ]


def selected_ids(labels):
    result = []
    for label in labels or []:
        if label.endswith("]") and "[" in label:
            result.append(label.rsplit("[", 1)[1][:-1])
    return result


def make_questions(context: PlanningContext):
    needs = build_question_needs(context)[:MAX_QUESTIONS_PER_ROUND]
    if not needs:
        return [], ""
    generated = question_chain.invoke({
        "context": context.model_dump_json(indent=2),
        "question_needs": [need.to_dict() for need in needs],
        "round_number": 1,
        "asked_keys": [],
    })
    need_keys = {need.key for need in needs}
    items = [item for item in generated.questions if item.key in need_keys]
    returned = {item.key for item in items}
    items.extend(fallback_question(need) for need in needs if need.key not in returned)
    blocks = ["### 계획을 만들기 위해 조금만 더 알려주세요"]
    for index, item in enumerate(items, 1):
        block = f"**{index}. {item.question}**"
        if item.example:
            block += f"\n\n> 예: {item.example.removeprefix('예:').strip()}"
        blocks.append(block)
    return [item.model_dump() for item in items], "\n\n---\n\n".join(blocks)


def render_question_data(items):
    blocks = ["### 계획을 만들기 위해 조금만 더 알려주세요"]
    for index, item in enumerate(items, 1):
        block = f"**{index}. {item['question']}**"
        if item.get("example"):
            block += f"\n\n> 예: {item['example'].removeprefix('예:').strip()}"
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)


def format_plan_markdown(context, plan, validation, title="생성된 계획"):
    lines = [f"## {title}"]
    strategy_guides = {
        "quick": "가능한 시간 중 빠른 완료를 우선했습니다.",
        "balanced": "계획 기간에 작업을 고르게 분산했습니다.",
        "buffer": "예상치 못한 변경을 위해 마지막 가용일을 비워 두었습니다.",
    }
    lines.append(f"\n> {strategy_guides[context.planning_strategy]} 표에서 직접 수정할 수 있습니다.")
    if plan.schedule:
        completion_date = max(item.date for item in plan.schedule)
        strategy_names = {
            "quick": "빠른 완료", "balanced": "마감일까지 균등 분산",
            "buffer": "마지막 가용일을 비워 두는 여유일 확보",
        }
        lines.append(
            f"\n\n- **계획 기간:** {context.start_date} ~ {context.end_date}"
            f"\n- **배치 전략:** {strategy_names[context.planning_strategy]}"
            f"\n- **예상 완료일:** {completion_date}"
        )
        if completion_date < context.end_date and all(
            not task.is_recurring for task in context.tasks
        ):
            lines.append(
                f"\n\n> 현재 입력한 일회성 작업은 {completion_date}까지 완료할 수 있어 "
                "계획 기간의 모든 날짜에 일정을 만들 필요가 없다고 판단했습니다."
            )

    if context.available_slots:
        lines.append("\n### 반영한 가용 시간")
        for slot in context.available_slots:
            if slot.date:
                label = slot.date
            elif sorted(slot.weekdays) == [0, 1, 2, 3, 4]:
                label = "평일"
            elif sorted(slot.weekdays) == [5, 6]:
                label = "주말"
            else:
                label = "·".join(WEEKDAY_NAMES[day] for day in slot.weekdays)
            if slot.start_time and slot.end_time:
                time_text = f"{slot.start_time}~{slot.end_time}"
            else:
                time_text = f"하루 {slot.available_minutes}분" if slot.available_minutes else "시간 미정"
            lines.append(f"\n- {label} · {time_text}")

    events = get_virtual_calendar_events.invoke({
        "context_json": context.model_dump_json()
    })
    if events:
        lines.append("\n### 반영한 고정 일정")
        for event in events:
            date_text = event["date"]
            if event.get("end_date") and event["end_date"] != event["date"]:
                date_text += f" ~ {event['end_date']}"
            time_text = (
                "종일" if event.get("is_all_day")
                else f"{event['start_time']}~{event['end_time']}"
            )
            lines.append(f"\n- {date_text} {time_text} · {event['title']}")

    lines.append("\n### 실행 일정")
    weekdays = ["월", "화", "수", "목", "금", "토", "일"]
    current = None
    for item in sorted(plan.schedule, key=lambda value: (value.date, value.start_time)):
        if item.date != current:
            current = item.date
            day = date.fromisoformat(item.date)
            lines.append(f"\n\n#### {item.date} ({weekdays[day.weekday()]})")
        lines.append(f"\n- **{item.start_time}~{item.end_time}** · {item.title}")
    if not plan.schedule:
        lines.append("\n- 배치된 작업이 없습니다.")

    if plan.warnings:
        lines.append(
            f"\n### 🚨 부분 계획: {len(plan.warnings)}개 작업을 배치하지 못했습니다."
        )
        warning_counts = Counter(plan.warnings)
        for warning, count in warning_counts.items():
            suffix = f" ({count}건)" if count > 1 else ""
            lines.append(f"\n- {warning}{suffix}")
        lines.append(
            "\n\n> 아래 **미배치 해결** 탭에서 여유일 사용, 기간 연장, "
            "추가 가능 시간 등록을 바로 실행할 수 있습니다."
        )
    lines.append(f"\n### 계획 이유\n\n{plan.explanation}")
    if not validation.is_valid:
        result = "충돌 발견"
    elif plan.warnings:
        result = "충돌은 없지만 일부 작업 미배치"
    else:
        result = "모든 작업 배치 완료"
    lines.append(f"\n### 가상 캘린더 검증: {result}")
    lines.extend(f"\n- {error}" for error in validation.errors)
    return "".join(lines)


def weekly_calendar_html(context, plan):
    if not context or not context.start_date or not context.end_date:
        return "<div class='calendar-empty'>계획을 생성하면 주간 캘린더가 표시됩니다.</div>"
    start, end = date.fromisoformat(context.start_date), date.fromisoformat(context.end_date)
    week_start = start - timedelta(days=start.weekday())
    task_map = {}
    for item in plan.schedule:
        task_map.setdefault(item.date, []).append(item)
    fixed = get_virtual_calendar_events.invoke({"context_json": context.model_dump_json()})
    fixed_map = {}
    for event in fixed:
        event_start = date.fromisoformat(event["date"])
        event_end = date.fromisoformat(event.get("end_date") or event["date"])
        cursor = event_start
        while cursor <= event_end:
            fixed_map.setdefault(cursor.isoformat(), []).append(event)
            cursor += timedelta(days=1)
    weekdays = ["월", "화", "수", "목", "금", "토", "일"]
    weeks = []
    while week_start <= end:
        days = []
        for offset in range(7):
            day = week_start + timedelta(days=offset)
            classes = "calendar-day"
            if day < start or day > end:
                classes += " outside"
            if day.weekday() in context.unavailable_weekdays:
                classes += " unavailable"
            cards = []
            if day.weekday() in context.unavailable_weekdays:
                cards.append("<div class='event-card unavailable-card'>계획 불가</div>")
            for event in fixed_map.get(day.isoformat(), []):
                time_text = (
                    "종일" if event.get("is_all_day")
                    else f"{event['start_time']}~{event['end_time']}"
                )
                cards.append(
                    f"<div class='event-card fixed-card'><b>{escape(event['title'])}</b>"
                    f"<small>{time_text}</small></div>"
                )
            for item in sorted(task_map.get(day.isoformat(), []), key=lambda value: value.start_time):
                cards.append(
                    f"<div class='event-card task-card'><b>{escape(item.title)}</b>"
                    f"<small>{item.start_time}~{item.end_time}</small></div>"
                )
            days.append(
                f"<div class='{classes}'><div class='day-header'>{weekdays[offset]} "
                f"<span>{day.month}/{day.day}</span></div>{''.join(cards)}</div>"
            )
        week_end = week_start + timedelta(days=6)
        weeks.append(
            f"<section class='calendar-week'><h3>{week_start:%m/%d} ~ {week_end:%m/%d}</h3>"
            f"<div class='week-grid'>{''.join(days)}</div></section>"
        )
        week_start += timedelta(days=7)
    warning = ""
    if plan.warnings:
        warning = (
            f"<div class='calendar-warning'><b>🚨 부분 계획</b><br>"
            f"{len(plan.warnings)}개 작업이 아직 미배치 상태입니다. "
            "아래 미배치 해결 탭에서 추가 가능 시간을 등록해 주세요.</div>"
        )
    return warning + "<div class='calendar-wrap'>" + "".join(weeks) + "</div>"


def generate_plan_output(context):
    plan, _ = create_plan(context, model)
    tool_result = check_virtual_calendar_conflicts.invoke({
        "context_json": context.model_dump_json(),
        "plan_json": plan.model_dump_json(),
    })
    validation = PlanValidation.model_validate(tool_result)
    return plan, validation, format_plan_markdown(context, plan, validation)


def analyze_request(user_input, strategy):
    empty_choices, empty_rows = empty_plan_controls()
    if not user_input.strip():
        return "요청을 입력해 주세요.", "", "", "", {}, [], "", {}, empty_choices, empty_rows
    context = extract_chain.invoke({
        "current_date": date.today().isoformat(), "user_input": user_input,
    })
    context = apply_text_constraints(context, user_input, date.today())
    context.planning_strategy = strategy
    context.missing_information = detect_missing_information(context)
    questions, markdown = make_questions(context)
    history = f"[최초 요청]\n{user_input}"
    if questions:
        return (
            "입력을 분석했습니다. 아래 질문에 답해주세요.", markdown, "", "",
            context.model_dump(), questions, history, {}, empty_choices, empty_rows,
        )
    context = apply_suggested_estimates(context)
    plan, _, output = generate_plan_output(context)
    return (
        "추가 질문 없이 계획을 생성했습니다.", "", output,
        weekly_calendar_html(context, plan),
        context.model_dump(), [], history, plan.model_dump(),
        gr.update(choices=plan_choices(plan), value=[]), plan_rows(plan),
    )


def apply_answers(answer_text, context_data, question_data, history, strategy):
    empty_choices, empty_rows = empty_plan_controls()
    if not context_data:
        return "먼저 요청 분석을 눌러주세요.", "", "", "", {}, [], history, {}, empty_choices, empty_rows
    if question_data and not answer_text.strip():
        return (
            "질문에 답해주세요.", render_question_data(question_data), "", "",
            context_data, question_data, history, {}, empty_choices, empty_rows,
        )
    context = PlanningContext.model_validate(context_data)
    paired = pair_questions_and_answers(
        [item["question"] for item in question_data], answer_text
    )
    history = f"{history}\n\n[질문과 답변]\n{paired}"
    merged = f"""{
        history
    }

[현재까지 구조화된 정보]
{context.model_dump_json(indent=2)}

최근 답변을 우선해 PlanningContext를 다시 작성한다.
답하지 않은 정보는 만들지 않고 충돌하지 않는 기존 정보는 유지한다.
질문과 답변은 순서대로 연결한다."""
    updated = extract_chain.invoke({
        "current_date": date.today().isoformat(), "user_input": merged,
    })
    updated = apply_text_constraints(updated, merged, date.today())
    updated.planning_strategy = strategy
    updated.missing_information = detect_missing_information(updated)
    questions, markdown = make_questions(updated)
    if questions:
        return (
            "아직 필요한 정보가 있습니다.", markdown, "", "", updated.model_dump(),
            questions, history, {}, empty_choices, empty_rows,
        )
    updated = apply_suggested_estimates(updated)
    plan, _, output = generate_plan_output(updated)
    return (
        "계획 생성과 검증을 완료했습니다.", "", output,
        weekly_calendar_html(updated, plan), updated.model_dump(),
        [], history, plan.model_dump(),
        gr.update(choices=plan_choices(plan), value=[]), plan_rows(plan),
    )


def build_comparison(previous, updated):
    old = {item.task_id: item for item in previous.schedule}
    new = {item.task_id: item for item in updated.schedule}
    blocks = []
    icons = {"유지": "✅", "이동": "🔄", "추가": "➕", "미배치": "⚠️"}
    for task_id in dict.fromkeys([*old, *new]):
        before, after = old.get(task_id), new.get(task_id)
        before_text = (
            f"{before.date} {before.start_time}~{before.end_time}" if before else "-"
        )
        after_text = (
            f"{after.date} {after.start_time}~{after.end_time}" if after else "-"
        )
        if before and after:
            state = "유지" if before_text == after_text else "이동"
            title = after.title
        elif after:
            state, title = "추가", after.title
        else:
            state, title = "미배치", before.title
        blocks.append(
            f"**{icons[state]} {state} · {title}**  \n"
            f"기존: {before_text} → 변경: {after_text}"
        )
    return "\n\n---\n\n".join(blocks)


def apply_replan(
    selected_labels, event_type, event_title,
    event_start_date, event_end_date, event_start_at, event_end_at,
    note, context_data, plan_data,
):
    if not context_data or not plan_data:
        return (
            "먼저 최초 계획을 생성해 주세요.", "", "", "",
            context_data, plan_data, gr.update(), [],
        )
    context = PlanningContext.model_validate(context_data)
    previous = Plan.model_validate(plan_data)
    missed_ids = selected_ids(selected_labels)
    new_events = []

    def unchanged_result(message):
        current_markdown = format_plan_markdown(
            context,
            previous,
            validate_plan(
                previous,
                build_free_slots(
                    context.available_slots, context.fixed_events,
                    context.start_date, context.end_date,
                ),
            ),
        )
        return (
            message, current_markdown, "",
            weekly_calendar_html(context, previous),
            context_data, plan_data, gr.update(), plan_rows(previous),
        )

    has_event_input = any([
        event_title, event_start_date, event_end_date, event_start_at, event_end_at,
    ])
    if has_event_input and event_type == "all_day":
        if not all([event_title, event_start_date, event_end_date]):
            return unchanged_result(
                "종일·기간 일정을 추가하려면 이름·시작 날짜·종료 날짜를 모두 입력해 주세요."
            )
        if event_end_date < event_start_date:
            return unchanged_result("종료 날짜는 시작 날짜보다 빠를 수 없습니다.")
        if (
            event_start_date.date() < date.fromisoformat(context.start_date)
            or event_end_date.date() > date.fromisoformat(context.end_date)
        ):
            return unchanged_result(
                f"새 일정은 현재 계획 기간({context.start_date}~{context.end_date}) 안에서만 "
                "추가할 수 있습니다. 먼저 미배치 해결 탭에서 계획 기간을 연장하거나 "
                "기간 안의 날짜를 선택해 주세요."
            )
        new_events.append(FixedEvent(
            id=f"ui-event-{len(context.fixed_events)+1}",
            title=event_title,
            date=event_start_date.date().isoformat(),
            end_date=event_end_date.date().isoformat(),
            is_all_day=True,
        ))
    elif has_event_input and event_type == "timed":
        if not all([event_title, event_start_at, event_end_at]):
            return unchanged_result(
                "시간 지정 일정을 추가하려면 이름·시작 일시·종료 일시를 모두 입력해 주세요."
            )
        if event_end_at <= event_start_at:
            return unchanged_result("새 일정의 종료 일시는 시작 일시보다 늦어야 합니다.")
        if (
            event_start_at.date() < date.fromisoformat(context.start_date)
            or event_end_at.date() > date.fromisoformat(context.end_date)
        ):
            return unchanged_result(
                f"새 일정은 현재 계획 기간({context.start_date}~{context.end_date}) 안에서만 "
                "추가할 수 있습니다. 먼저 미배치 해결 탭에서 계획 기간을 연장하거나 "
                "기간 안의 일시를 선택해 주세요."
            )
        new_events.append(FixedEvent(
            id=f"ui-event-{len(context.fixed_events)+1}",
            title=event_title,
            date=event_start_at.date().isoformat(),
            end_date=(event_end_at.date().isoformat() if event_end_at.date() != event_start_at.date() else None),
            is_all_day=False,
            start_time=event_start_at.strftime("%H:%M"),
            end_time=event_end_at.strftime("%H:%M"),
        ))

    missed_ids = list(dict.fromkeys(missed_ids))
    if not missed_ids:
        return unchanged_result(
            "체크박스에서 완료하지 못한 작업을 하나 이상 선택해 주세요."
        )

    updated, validation = replan(
        context, previous, missed_ids, new_events, model
    )
    display_context = context.model_copy(deep=True)
    display_context.fixed_events.extend(new_events)
    comparison = "## 기존 계획과 변경 계획 비교\n\n" + build_comparison(
        previous, updated
    )
    output = comparison + "\n\n" + format_plan_markdown(
        display_context, updated, validation, "재계획 결과"
    )
    if note.strip():
        output += f"\n\n### 사용자가 남긴 재계획 메모\n\n{note.strip()}"
    if updated.warnings:
        status = (
            f"재계획했지만 가용 시간이 부족해 {len(updated.warnings)}개 작업을 "
            "배치하지 못했습니다. 아래 해결 방법을 확인해 주세요."
        )
    else:
        status = "선택한 미완료 작업과 새 일정을 반영했습니다."
    return (
        status, output, output, weekly_calendar_html(display_context, updated),
        display_context.model_dump(), updated.model_dump(),
        gr.update(choices=plan_choices(updated), value=[]),
        plan_rows(updated),
    )


def apply_plan_edits(rows, context_data, plan_data):
    if not context_data or not plan_data:
        return "먼저 계획을 생성해 주세요.", "", "", plan_data, gr.update(), rows
    values = rows.values.tolist() if hasattr(rows, "values") else rows
    try:
        schedule = []
        for task_id, title, day, start, end in values:
            start_dt = datetime.fromisoformat(f"{str(day)[:10]}T{start}")
            end_dt = datetime.fromisoformat(f"{str(day)[:10]}T{end}")
            minutes = int((end_dt - start_dt).total_seconds() // 60)
            if minutes <= 0:
                raise ValueError(f"{title}: 종료 시간은 시작 시간보다 늦어야 합니다.")
            schedule.append(ScheduleItem(
                task_id=str(task_id), title=str(title), date=str(day)[:10],
                start_time=str(start), end_time=str(end), minutes=minutes,
            ))
    except Exception as error:
        return f"수정한 표를 확인해 주세요: {error}", "", "", plan_data, gr.update(), rows

    context = PlanningContext.model_validate(context_data)
    previous = Plan.model_validate(plan_data)
    edited = previous.model_copy(deep=True)
    edited.schedule = sorted(schedule, key=lambda item: (item.date, item.start_time))
    free = build_free_slots(
        context.available_slots, context.fixed_events,
        context.start_date, context.end_date,
    )
    validation = validate_plan(edited, free)
    output = "## 수정 전후 비교\n\n" + build_comparison(previous, edited)
    output += "\n\n" + format_plan_markdown(
        context, edited, validation, "사용자가 수정한 계획"
    )
    status = (
        "수정 내용을 저장했고 검증을 통과했습니다."
        if validation.is_valid else
        "수정 내용은 저장했지만 충돌이 있습니다. 검증 결과를 확인해 주세요."
    )
    return (
        status, output, weekly_calendar_html(context, edited), edited.model_dump(),
        gr.update(choices=plan_choices(edited), value=[]), plan_rows(edited),
    )


def toggle_event_type(event_type):
    is_all_day = event_type == "all_day"
    return (
        gr.update(visible=is_all_day),
        gr.update(visible=is_all_day),
        gr.update(visible=not is_all_day),
        gr.update(visible=not is_all_day),
    )


def resolution_result(context, message):
    plan, _, markdown = generate_plan_output(context)
    remaining = len(plan.warnings)
    status = (
        f"{message} 모든 작업을 배치했습니다."
        if remaining == 0 else
        f"{message} 다시 계산했지만 {remaining}개 작업은 아직 미배치 상태입니다."
    )
    return (
        status, markdown, markdown, weekly_calendar_html(context, plan),
        context.model_dump(), plan.model_dump(),
        gr.update(choices=plan_choices(plan), value=[]), plan_rows(plan),
    )


def use_reserved_day(context_data, plan_data):
    if not context_data or not plan_data:
        return "먼저 계획을 생성해 주세요.", "", "", "", context_data, plan_data, gr.update(), []
    context = PlanningContext.model_validate(context_data)
    context.planning_strategy = "balanced"
    return resolution_result(context, "비워 두었던 마지막 가용일을 사용해")


def extend_unplaced_period(new_end_at, context_data, plan_data):
    if not context_data or not plan_data:
        return "먼저 계획을 생성해 주세요.", "", "", "", context_data, plan_data, gr.update(), []
    if not new_end_at:
        return "새 계획 종료 날짜를 선택해 주세요.", "", "", "", context_data, plan_data, gr.update(), []
    context = PlanningContext.model_validate(context_data)
    previous_plan = Plan.model_validate(plan_data)
    try:
        updated_context, updated_plan, validation = extend_unplaced_tasks(
            context,
            previous_plan,
            new_end_at.date().isoformat(),
            model,
        )
    except ValueError as error:
        return (
            str(error), "", "", weekly_calendar_html(context, previous_plan),
            context_data, plan_data, gr.update(), plan_rows(previous_plan),
        )
    markdown = format_plan_markdown(updated_context, updated_plan, validation)
    remaining = len(updated_plan.unscheduled_tasks)
    status = (
        "기존 일정과 반복 횟수는 유지하고 미배치 작업만 연장 구간에 배치했습니다."
        if remaining == 0 else
        f"연장 구간에 다시 배치했지만 {remaining}개 작업은 아직 미배치 상태입니다."
    )
    return (
        status, markdown, markdown,
        weekly_calendar_html(updated_context, updated_plan),
        updated_context.model_dump(), updated_plan.model_dump(),
        gr.update(choices=plan_choices(updated_plan), value=[]),
        plan_rows(updated_plan),
    )


def add_extra_availability(extra_start, extra_end, context_data, plan_data):
    if not context_data or not plan_data:
        return "먼저 계획을 생성해 주세요.", "", "", "", context_data, plan_data, gr.update(), []
    if not extra_start or not extra_end:
        return "추가로 가능한 시작·종료 일시를 모두 선택해 주세요.", "", "", "", context_data, plan_data, gr.update(), []
    if extra_end <= extra_start:
        return "추가 가능 시간의 종료는 시작보다 늦어야 합니다.", "", "", "", context_data, plan_data, gr.update(), []
    if extra_start.date() != extra_end.date():
        return "추가 가능 시간은 같은 날짜 안에서 선택해 주세요.", "", "", "", context_data, plan_data, gr.update(), []
    context = PlanningContext.model_validate(context_data)
    context.available_slots.append(AvailableSlot(
        id=f"extra-{len(context.available_slots)+1}",
        date=extra_start.date().isoformat(),
        start_time=extra_start.strftime("%H:%M"),
        end_time=extra_end.strftime("%H:%M"),
    ))
    return resolution_result(context, "추가 가능 시간을 반영해")


CSS = """
.gradio-container {max-width: 1180px !important; margin: auto !important;}
.hero {padding: 24px; border-radius: 18px; background: linear-gradient(135deg,#eef2ff,#f5f3ff);}
.hero, .hero h1, .hero p {color: #1f2937 !important;}
.section-card {border: 1px solid #e5e7eb; border-radius: 16px; padding: 10px;}
#status-box {border-left: 4px solid #6366f1; padding-left: 14px;}
.calendar-wrap {display:flex; flex-direction:column; gap:20px;}
.calendar-warning {margin-bottom:14px; padding:14px; border-radius:12px; background:#fef2f2; color:#991b1b; border:1px solid #fecaca;}
.calendar-week h3 {margin:0 0 8px; font-size:15px; color:#4f46e5;}
.week-grid {display:grid; grid-template-columns:repeat(7,minmax(110px,1fr)); gap:7px; overflow-x:auto;}
.calendar-day {min-height:130px; padding:8px; border:1px solid #e5e7eb; border-radius:12px; background:#fff;}
.calendar-day.outside {opacity:.35;}
.calendar-day.unavailable {background:#fef2f2;}
.day-header {font-weight:700; margin-bottom:7px; display:flex; justify-content:space-between;}
.day-header span {font-weight:400; color:#6b7280;}
.event-card {padding:7px; border-radius:8px; margin:5px 0; font-size:12px;}
.event-card small {display:block; margin-top:3px;}
.task-card {background:#ede9fe; color:#4c1d95; border-left:3px solid #7c3aed;}
.fixed-card {background:#f3f4f6; color:#374151; border-left:3px solid #6b7280;}
.unavailable-card {background:#fee2e2; color:#991b1b;}
@media (prefers-color-scheme: dark) {
  .hero {background: linear-gradient(135deg,#1e1b4b,#312e81);}
  .hero, .hero h1, .hero p {color: #f9fafb !important;}
  .section-card {border-color: #374151;}
  .calendar-day {background:#111827; border-color:#374151;}
  .calendar-day.unavailable {background:#3f1d25;}
  .task-card {background:#312e81; color:#ede9fe;}
  .fixed-card {background:#374151; color:#f3f4f6;}
}
@media (max-width: 800px) {.week-grid {grid-template-columns:repeat(7,150px);}}
"""
THEME = gr.themes.Soft(primary_hue="indigo", secondary_hue="violet")

with gr.Blocks(title="AI 동적 플래너") as demo:
    gr.Markdown(
        "# AI 동적 플래너\n해야 할 일을 말하면 필요한 조건만 확인하고 실행 가능한 일정으로 정리합니다.",
        elem_classes=["hero"],
    )
    context_state, question_state = gr.State({}), gr.State([])
    history_state, plan_state = gr.State(""), gr.State({})

    with gr.Row():
        with gr.Column(scale=5, elem_classes=["section-card"]):
            gr.Markdown("## 1. 계획 요청")
            request_input = gr.Textbox(
                label="어떤 계획을 세우고 싶나요?", lines=6,
                placeholder="예: 이번 주 일요일까지 LangChain 프로젝트를 완성하고 싶어...",
            )
            strategy_input = gr.Radio(
                choices=[
                    ("빠르게 끝내기", "quick"),
                    ("마감일까지 균등 분산", "balanced"),
                    ("마지막 가용일을 비워 두기", "buffer"),
                ],
                value="quick",
                label="계획 방식",
            )
            analyze_button = gr.Button("요청 분석", variant="primary")
            status_output = gr.Markdown(elem_id="status-box")
            question_output = gr.Markdown()
            answer_input = gr.Textbox(
                label="추가 질문 답변", lines=5,
                placeholder="질문 번호 순서대로 한 줄씩 답해주세요.",
            )
            create_button = gr.Button("답변 반영하고 계획 만들기", variant="primary")

        with gr.Column(scale=7, elem_classes=["section-card"]):
            gr.Markdown("## 2. 계획 결과")
            with gr.Tabs():
                with gr.Tab("주간 캘린더"):
                    calendar_output = gr.HTML(
                        "<div class='calendar-empty'>계획을 생성하면 주간 캘린더가 표시됩니다.</div>"
                    )
                with gr.Tab("날짜별 상세"):
                    plan_output = gr.Markdown()

    with gr.Tabs():
        with gr.Tab("미배치 해결"):
            gr.Markdown("""
            ### 미배치 작업을 바로 다시 계획하기
            계획 결과에 `부분 계획` 경고가 있을 때 아래 방법 중 하나를 실행하세요.
            실행 후 계획과 주간 캘린더가 즉시 갱신됩니다.
            """)
            use_buffer_button = gr.Button(
                "비워 둔 여유일 사용", variant="primary"
            )
            with gr.Group():
                gr.Markdown(
                    "**기간을 늘려 해결하려면**  \n"
                    "기존 일정과 반복 작업 횟수는 바꾸지 않고, 현재 미배치 작업만 "
                    "기존 종료일 다음 날부터 새 종료일까지 배치합니다."
                )
                with gr.Row():
                    extended_end_date = gr.DateTime(
                        label="새 계획 종료 날짜",
                        include_time=False, type="datetime", timezone="Asia/Seoul",
                    )
                    extend_button = gr.Button(
                        "미배치 작업만 연장 구간에 배치", variant="secondary"
                    )
            with gr.Group():
                gr.Markdown("**특정 날짜에 추가로 시간을 낼 수 있다면**")
                with gr.Row():
                    extra_start = gr.DateTime(
                        label="추가 가능 시작 일시",
                        include_time=True, type="datetime", timezone="Asia/Seoul",
                    )
                    extra_end = gr.DateTime(
                        label="추가 가능 종료 일시",
                        include_time=True, type="datetime", timezone="Asia/Seoul",
                    )
                    add_time_button = gr.Button("추가 시간 반영")
            resolution_output = gr.Markdown()

        with gr.Tab("재계획"):
            gr.Markdown("계획대로 끝내지 못한 작업을 선택하세요. 새 일정 입력은 선택 사항입니다.")
            missed_selector = gr.CheckboxGroup(
                label="계획대로 완료하지 못한 작업", choices=[],
                info="체크한 작업과 그 이후 일정만 다시 배치합니다.",
            )
            event_type = gr.Radio(
                choices=[
                    ("종일·기간 일정", "all_day"),
                    ("시간 지정 일정", "timed"),
                ],
                value="all_day",
                label="새 일정 유형(선택)",
                info="출장·여행처럼 날짜 범위 전체가 불가능하면 종일·기간 일정을 선택합니다.",
            )
            with gr.Row():
                event_title = gr.Textbox(
                    label="새 일정 이름(선택)",
                    placeholder="예: 출장, 회의",
                    info="새 일정을 추가할 때만 날짜 또는 일시와 함께 입력합니다.",
                )
                event_start_date = gr.DateTime(
                    label="시작 날짜(선택)", include_time=False,
                    type="datetime", timezone="Asia/Seoul",
                )
                event_end_date = gr.DateTime(
                    label="종료 날짜(선택)", include_time=False,
                    type="datetime", timezone="Asia/Seoul",
                    info="시작일부터 종료일까지 모든 가용 시간을 막습니다.",
                )
                event_start_at = gr.DateTime(
                    label="시작 일시(선택)", include_time=True,
                    type="datetime", timezone="Asia/Seoul", visible=False,
                )
                event_end_at = gr.DateTime(
                    label="종료 일시(선택)", include_time=True,
                    type="datetime", timezone="Asia/Seoul",
                    visible=False,
                )
            replan_note = gr.Textbox(
                label="재계획 메모(선택)",
                placeholder="예: 병원 일정 때문에 이번 주에는 시간이 부족함",
                info="메모는 결과에 함께 표시되며 일정이나 작업으로 자동 변환되지 않습니다.",
            )
            replan_button = gr.Button("변경 사항 반영해 재계획", variant="primary")
            replan_output = gr.Markdown()

        with gr.Tab("계획 직접 수정"):
            gr.Markdown("날짜와 시간을 수정한 뒤 저장하면 충돌을 다시 검사합니다.")
            plan_editor = gr.Dataframe(
                headers=["task_id", "작업", "날짜", "시작", "종료"],
                datatype=["str", "str", "str", "str", "str"],
                interactive=True,
                row_count=(0, "dynamic"),
            )
            edit_button = gr.Button("수정 내용 저장 및 검증")
            edit_output = gr.Markdown()

    common_outputs = [
        status_output, question_output, plan_output, calendar_output,
        context_state, question_state, history_state, plan_state,
        missed_selector, plan_editor,
    ]
    analyze_button.click(
        analyze_request, [request_input, strategy_input], common_outputs
    )
    create_button.click(
        apply_answers,
        [answer_input, context_state, question_state, history_state, strategy_input],
        common_outputs,
    )
    replan_button.click(
        apply_replan,
        [
            missed_selector, event_type, event_title,
            event_start_date, event_end_date, event_start_at, event_end_at,
            replan_note, context_state, plan_state,
        ],
        [
            status_output, plan_output, replan_output, calendar_output,
            context_state, plan_state, missed_selector, plan_editor,
        ],
    )
    event_type.change(
        toggle_event_type,
        inputs=[event_type],
        outputs=[event_start_date, event_end_date, event_start_at, event_end_at],
    )
    edit_button.click(
        apply_plan_edits,
        [plan_editor, context_state, plan_state],
        [status_output, edit_output, calendar_output, plan_state, missed_selector, plan_editor],
    )
    resolution_outputs = [
        status_output, plan_output, resolution_output, calendar_output,
        context_state, plan_state, missed_selector, plan_editor,
    ]
    use_buffer_button.click(
        use_reserved_day,
        [context_state, plan_state],
        resolution_outputs,
    )
    extend_button.click(
        extend_unplaced_period,
        [extended_end_date, context_state, plan_state],
        resolution_outputs,
    )
    add_time_button.click(
        add_extra_availability,
        [extra_start, extra_end, context_state, plan_state],
        resolution_outputs,
    )


In [12]:
demo.launch(theme=THEME, css=CSS)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6cca309f8ddcf38b15.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
